# Feature engineering - advanced data preparation pipeline  | Sebislaw

## Libraries

In [4]:
from os.path  import join
import random
import itertools
import math

import numpy as np
import pandas as pd
from pandas.plotting import scatter_matrix

import matplotlib.pyplot as plt
import plotly.express as px
from pandas.plotting import parallel_coordinates
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display

from sklearn.linear_model import LinearRegression, LassoCV, LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score, StratifiedKFold, RandomizedSearchCV
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_classif
from sklearn.neural_network import MLPClassifier

import xgboost as xgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import optuna
# from tabpfn import TabPFNClassifier

## Data

In [5]:
data_path = '..\\..\\..\\data'
pd.set_option('display.max_columns', None)

# The Basics ------------------------------------------------------------------------
# Men
MTeams = pd.read_csv(join(data_path, 'MTeams.csv'))
MSeasons = pd.read_csv(join(data_path, 'MSeasons.csv'))
MNCAATourneySeeds = pd.read_csv(join(data_path, 'MNCAATourneySeeds.csv'))
MRegularSeasonCompactResults = pd.read_csv(join(data_path, 'MRegularSeasonCompactResults.csv'))
MNCAATourneyCompactResults = pd.read_csv(join(data_path, 'MNCAATourneyCompactResults.csv'))
# Women
WTeams = pd.read_csv(join(data_path, 'WTeams.csv'))
WSeasons = pd.read_csv(join(data_path, 'WSeasons.csv'))
WNCAATourneySeeds = pd.read_csv(join(data_path, 'WNCAATourneySeeds.csv'))
WRegularSeasonCompactResults = pd.read_csv(join(data_path, 'WRegularSeasonCompactResults.csv'))
WNCAATourneyCompactResults = pd.read_csv(join(data_path, 'WNCAATourneyCompactResults.csv'))
# Other
SampleSubmissionStage1 = pd.read_csv(join(data_path, 'SampleSubmissionStage1.csv'))
SampleSubmissionStage2 = pd.read_csv(join(data_path, 'SampleSubmissionStage2.csv'))
SeedBenchmarkStage1 = pd.read_csv(join(data_path, 'SeedBenchmarkStage1.csv'))

# Team Box Scores ------------------------------------------------------------------------
# Men
MRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'MRegularSeasonDetailedResults.csv'))
MNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'MNCAATourneyDetailedResults.csv'))
# Women
WRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'WRegularSeasonDetailedResults.csv'))
WNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'WNCAATourneyDetailedResults.csv'))

# Geography ------------------------------------------------------------------------
# All
Cities = pd.read_csv(join(data_path, 'Cities.csv'))
Conferences = pd.read_csv(join(data_path, 'Conferences.csv'))
# Men
MGameCities = pd.read_csv(join(data_path, 'MGameCities.csv'))
# Women
WGameCities = pd.read_csv(join(data_path, 'WGameCities.csv'))

# Public Rankings ------------------------------------------------------------------------
# Men
MMasseyOrdinals = pd.read_csv(join(data_path, 'MMasseyOrdinals.csv')) # men only

# Supplements ------------------------------------------------------------------------
# Men
MTeamCoaches = pd.read_csv(join(data_path, 'MTeamCoaches.csv')) # men only
MTeamConferences = pd.read_csv(join(data_path, 'MTeamConferences.csv'))
MConferenceTourneyGames = pd.read_csv(join(data_path, 'MConferenceTourneyGames.csv'))
MSecondaryTourneyTeams = pd.read_csv(join(data_path, 'MSecondaryTourneyTeams.csv'))
MSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'MSecondaryTourneyCompactResults.csv'))
MTeamSpellings = pd.read_csv(join(data_path, "MTeamSpellings.csv"), encoding='cp1252')
MNCAATourneySlots = pd.read_csv(join(data_path, 'MNCAATourneySlots.csv'))
MNCAATourneySeedRoundSlots = pd.read_csv(join(data_path, 'MNCAATourneySeedRoundSlots.csv')) # men only
# Women
WTeamConferences = pd.read_csv(join(data_path, 'WTeamConferences.csv'))
WConferenceTourneyGames = pd.read_csv(join(data_path, 'WConferenceTourneyGames.csv'))
WSecondaryTourneyTeams = pd.read_csv(join(data_path, 'WSecondaryTourneyTeams.csv'))
WSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'WSecondaryTourneyCompactResults.csv'))
WTeamSpellings = pd.read_csv(join(data_path, 'WTeamSpellings.csv'), encoding='cp1252')
WNCAATourneySlots = pd.read_csv(join(data_path, 'WNCAATourneySlots.csv'))

## Data preparation pipeline

In [6]:
def prepare_data(df):
        
    """
    This function duplicates and flips a game record.
    Now two records with the same data are present, 
    but viewed from perspectives of two different teams.
    """
    
    df = df[[
         'Season', 'DayNum', 'NumOT',
         'WTeamID',  'WScore', 'WLoc',
         'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF',
         'LTeamID', 'LScore',
         'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF'
    ]]
    dfswap = df[[
         'Season', 'DayNum', 'NumOT',
         'LTeamID', 'LScore', 'WLoc',
         'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF',
         'WTeamID',  'WScore',
         'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF',
    ]].copy()
    
    dfswap.loc[df['WLoc'] == 'H', 'WLoc'] = 'A'
    dfswap.loc[df['WLoc'] == 'A', 'WLoc'] = 'H'
        
    df = df.rename(columns={'WLoc': 'location'})
    dfswap = dfswap.rename(columns={'WLoc': 'location'})
        
    df.columns = [x.replace('W','T1_').replace('L','T2_') for x in list(df.columns)]
    dfswap.columns = [x.replace('L','T1_').replace('W','T2_') for x in list(dfswap.columns)]
    
    output = pd.concat([df, dfswap]).reset_index(drop=True)
    output.loc[output.location=='N','location'] = '0'
    output.loc[output.location=='H','location'] = '1'
    output.loc[output.location=='A','location'] = '-1'
    output.location = output.location.astype(int)
        
    output['PointDiff'] = output['T1_Score'] - output['T2_Score']
    
    return output

def get_data(regular_results, tourney_results, seeds, prepared=False, location_multiplier=[1, 1], win_ratio_days_back=14):

    """
    This function uses the prepare_data function in order to create
    a data frame with season statistics for each team.
    These statistics are added to records with games played in
    tournament to make data 'x' used in model to predict the game 
    result 'y'. The output is a data frame that contains data 'x'
    and also label 'y' can be easily calculated based on score difference in matches.
    """

    if prepared:
        regular_data = regular_results.copy()
        tourney_data = tourney_results.copy()
    else:
        # make data frames with extra rows to represent the perspective of losing team
        regular_data = prepare_data(regular_results)
        tourney_data = prepare_data(tourney_results)

    # ----------------------------------- Add reward/penalty for playing in home or away
    if location_multiplier[0] == 1 and location_multiplier[1] == 1:
        # data frame with mean game statistics for a given team in a given season
        season_statistics  = regular_data.groupby(["Season", 'T1_TeamID'])[
            [
                'T1_Score', 'T1_FGM','T1_FGA','T1_FGM3','T1_FGA3','T1_FTM','T1_FTA',
                'T1_OR','T1_DR','T1_Ast','T1_TO','T1_Stl','T1_Blk','T1_PF',
                'T2_Score', 'T2_FGM','T2_FGA','T2_FGM3','T2_FGA3','T2_FTM','T2_FTA',
                'T2_OR','T2_DR','T2_Ast','T2_TO','T2_Stl','T2_Blk','T2_PF',
                'PointDiff'
            ]
        ].agg('mean').reset_index()
    else:
        # Define which columns to adjust (you can add or remove columns as needed)
        T1_cols = ['T1_Score','T1_FGM','T1_FGA','T1_FGM3','T1_FGA3','T1_FTM','T1_FTA','T1_OR','T1_DR','T1_Ast','T1_TO','T1_Stl','T1_Blk','T1_PF']
        T2_cols = ['T2_Score','T2_FGM','T2_FGA','T2_FGM3','T2_FGA3','T2_FTM','T2_FTA','T2_OR','T2_DR','T2_Ast','T2_TO','T2_Stl','T2_Blk','T2_PF']
    
        # Convert the relevant columns to float before applying the adjustment function.
        cols_to_float = T1_cols + T2_cols
        regular_data[cols_to_float] = regular_data[cols_to_float].astype(float)
        
        def adjust_stats(row):
            # Determine multipliers based on location
            if row['location'] == 1:
                factor_T1 = location_multiplier[0]  # penalize Team1 stats (home)
                factor_T2 = location_multiplier[1]  # boost Team2 stats
            elif row['location'] == -1:
                factor_T1 = location_multiplier[1]  # boost Team1 stats (away)
                factor_T2 = location_multiplier[0]  # penalize Team2 stats
            else:
                factor_T1 = 1.0
                factor_T2 = 1.0
        
            # Adjust Team1 stats
            for col in T1_cols:
                if col in row and pd.notnull(row[col]):
                    row[col] = row[col] * factor_T1
        
            # Adjust Team2 stats
            for col in T2_cols:
                if col in row and pd.notnull(row[col]):
                    row[col] = row[col] * factor_T2
        
            # Recalculate derived statistics (if needed)
            if 'T1_Score' in row and 'T2_Score' in row:
                row['PointDiff'] = row['T1_Score'] - row['T2_Score']
            return row
        
        # Apply the adjustment function row-wise.
        regular_data_adjusted = regular_data.apply(adjust_stats, axis=1)
        
        # Now group by Season and T1_TeamID to compute season averages for the adjusted statistics.
        stats_columns = T1_cols[1:] + T2_cols[1:] + ['PointDiff']  # Exclude T1_TeamID from stats if present.
        season_statistics = regular_data_adjusted.groupby(["Season", 'T1_TeamID'])[stats_columns].agg('mean').reset_index()
    # -----------------------------------
    
    # mean statistics for team and team's opponent's
    season_statistics_T1 = season_statistics.copy()
    season_statistics_T2 = season_statistics.copy()
    
    season_statistics_T1.columns = ["T1_" + x.replace("T1_","").replace("T2_","opponent_") for x in list(season_statistics_T1.columns)]
    season_statistics_T2.columns = ["T2_" + x.replace("T1_","").replace("T2_","opponent_") for x in list(season_statistics_T2.columns)]
    season_statistics_T1.columns.values[0] = "Season"
    season_statistics_T2.columns.values[0] = "Season"
    season_statistics_T1 = season_statistics_T1.rename(columns={'T1_Score': 'T1_Score_mean'})
    season_statistics_T2 = season_statistics_T2.rename(columns={'T2_Score': 'T2_Score_mean'})
    
    # data frame containing game's result
    tourney_data = tourney_data[['Season', 'DayNum', 'T1_TeamID', 'T1_Score', 'T2_TeamID' ,'T2_Score', 'location']]
    tourney_data = pd.merge(tourney_data, season_statistics_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, season_statistics_T2, on = ['Season', 'T2_TeamID'], how = 'left')

    calculate_win_ratio_days_back = 132 - win_ratio_days_back
    
    # data frame with win fraction from last x days for a given team in a given season
    last14days_stats_T1 = regular_data.loc[regular_data.DayNum>calculate_win_ratio_days_back].reset_index(drop=True)
    last14days_stats_T1['win'] = np.where(last14days_stats_T1['PointDiff']>0,1,0)
    last14days_stats_T1 = last14days_stats_T1.groupby(['Season','T1_TeamID'])['win'].mean().reset_index(name='T1_win_ratio_14d')
    
    last14days_stats_T2 = regular_data.loc[regular_data.DayNum>calculate_win_ratio_days_back].reset_index(drop=True)
    last14days_stats_T2['win'] = np.where(last14days_stats_T2['PointDiff']<0,1,0)
    last14days_stats_T2 = last14days_stats_T2.groupby(['Season','T2_TeamID'])['win'].mean().reset_index(name='T2_win_ratio_14d')
    
    # add to tourney_data column with win fraction for winning and losing team
    tourney_data = pd.merge(tourney_data, last14days_stats_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, last14days_stats_T2, on = ['Season', 'T2_TeamID'], how = 'left')
    
    # get seeds with no regional division
    seeds['seed'] = seeds['Seed'].apply(lambda x: int(x[1:3]))
    
    # give each team a raw seed
    seeds_T1 = seeds[['Season','TeamID','seed']].copy()
    seeds_T2 = seeds[['Season','TeamID','seed']].copy()
    seeds_T1.columns = ['Season','T1_TeamID','T1_seed']
    seeds_T2.columns = ['Season','T2_TeamID','T2_seed']
    
    # add seeds to turney data for team 1 and team 2
    tourney_data = pd.merge(tourney_data, seeds_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, seeds_T2, on = ['Season', 'T2_TeamID'], how = 'left')
    
    # add a seed difference column
    tourney_data["Seed_diff"] = tourney_data["T1_seed"] - tourney_data["T2_seed"]

    return tourney_data

def get_df(seeds,
            season_games, season_range, days_back,
            tourney_games, tourney_range, 
            location_multiplier=[1, 1],
           win_ratio_days_back=14):

    """
    This function uses get_data function to get a data
    frame which is then used to make 'x' and 'y' data
    used in models.
    """
    
    # Get from regular season games from specified season range and days back
    # The results of those games will be used as team statictics (additional team information for tourney games)
    regular_results = season_games[season_games['Season'].isin(season_range)]
    regular_results = regular_results[regular_results['DayNum'] > 134 - days_back]
    
    # Get tourney games from a specified season
    # The results of those games will be used as labels
    tourney_results = tourney_games[tourney_games['Season'].isin(tourney_range)]
    
    # Get final data frame
    df = get_data(regular_results, tourney_results, seeds, location_multiplier=[1, 1], win_ratio_days_back=win_ratio_days_back)
    
    return df

def get_final_df(seeds,
                   season_games, season_range, days_back,
                   tourney_games, tourney_range, 
                   SampleSubmissionStage1,
                  location_multiplier=[1, 1],
                win_ratio_days_back=14):

    """
    This function works the same as function get_x_y,
    but also creates (at the moment it's the same as get_x_y)
    aditional data points mostly with NaN values that matches
    the submission format (parsed team matchups with the sample submission file).
    """

    # Get from regular season games from specified season range and days back
    # The results of those games will be used as team statictics (additional team information for tourney games)
    regular_results = season_games[season_games['Season'].isin(season_range)]
    regular_results = regular_results[regular_results['DayNum'] > 134 - days_back]

    # Get tourney games from a specified season
    # The results of those games will be used as labels
    tourney_results = tourney_games[tourney_games['Season'].isin(tourney_range)]
    
    # Assume sample_submission is a DataFrame with an "ID" column like "2023_1101_1102"
    # and tourney_results is a DataFrame with columns including: Season, WTeamID, LTeamID, DayNum, WScore, LScore, WLoc, etc.
    # Filter rows where the ID starts with the specified season (followed by an underscore)
    final_season = tourney_range[0]
    sample_submission_copy = SampleSubmissionStage1.copy()
    sample_submission = sample_submission_copy[sample_submission_copy['ID'].str.startswith(f"{final_season}_")]
    sample_submission = sample_submission.drop(columns=['Pred'])
    
    sample_submission[['Season', 'Team1', 'Team2']] = sample_submission['ID'].str.split('_', expand=True)
    sample_submission['Season'] = sample_submission['Season'].astype(float)
    sample_submission['Team1'] = sample_submission['Team1'].astype(float)
    sample_submission['Team2'] = sample_submission['Team2'].astype(float)
    
    regular_data_final = prepare_data(regular_results)
    tourney_data_final  = prepare_data(tourney_results)
    
    tourney_data =  get_data(regular_data_final, tourney_data_final,
                             seeds, prepared=True,
                          win_ratio_days_back=win_ratio_days_back)
    tourney_data = pd.merge(
        sample_submission,
        tourney_data,
        left_on=['Season', 'Team1', 'Team2'],
        right_on=['Season', 'T1_TeamID', 'T2_TeamID'],
        how='left'
    )
    tourney_data['T1_TeamID'] = tourney_data['Team1']
    tourney_data['T2_TeamID'] = tourney_data['Team2']
    tourney_data = tourney_data.drop(['ID', 'Team1', 'Team2'], axis=1)
    
    return tourney_data

def correct_predictions_based_on_seed(x, y, maximum_favoured_seed = 4, number_of_added_columns=1):
    """
    Sets the winnning chance to 1 or 0 based on the seed difference.
    """
    for i in range(len(x)):
        if x[i][-1-number_of_added_columns] <= (-16 + maximum_favoured_seed * 2 - 1):
            y[i] = 1
        elif x[i][-1-number_of_added_columns] >= (16 - maximum_favoured_seed * 2 + 1):
            y[i] = 0
    return y

def clear_na_from_x_y(x, y):
    """
    The data frame for final season is in format matching the submission file.
    This function clears NaNs from data.
    """
    # Create masks for training data:
    mask_train = ~np.isnan(x).any(axis=1) & ~np.isnan(y)
    x_clean = x[mask_train]
    y_clean = y[mask_train]
    return x_clean, y_clean

def x_y_from_data_frame(df):
    # Prepare data and labels
    x = df[list(df.columns[7:])].values
    y = np.where(
        df[['T1_Score', 'T2_Score']].isnull().any(axis=1),
        np.nan,
        np.where(df['T1_Score'] - df['T2_Score'] > 0, 1, 0)
    )
    return x, y

def get_all_core_data(
    regular_results, tourney_results, seeds, SampleSubmissionStage1,
    final_season = 2024, # the season we want to predict, so for out submission it will be 2025
    start_season = 2005, # from which ponit should we begin creating data
    season_years_list  = [[i-1, i] for i in range(2005, 2024+1)], # at which seasons to look at when calculating team's stats
    days_back = 15, # how many days back from the start of tourney to calculate team's stats per season
    maximum_favoured_seed = 4, # Set the predicted probability of winning to 1 for seeds <= maximum_favoured_seed and to 0 for >= 16-maximum_favoured_seed
    location_multiplier=[0.95, 1.05], # home penalty, away bonus 
    include_men = True, # include M... data sets when preparing x and y
    include_women = True, # include W... data sets when preparing x and y
    win_ratio_days_back = 14):

    """
    The idea of this function is to easily get data needed to train and test the model later on, with minimal code
    to not clutter the netebook.
    
    This function outputs df, x, y, df_final, x_final_season, y_final_season.
    
    df is a data frame with first 6 columns from tourney games and other calculated from other data frames.
    
    Adding a column to df and executing x_y_from_data_frame(df) function will yield x with added data.
    
    Note that df_final has the same structure as df, but also with rows with NaNs. The rows with missing information are there
    to match the sumbission file format. The separation of those data frames is to ensure that information from last season doesn't
    leak into training data due to poorly written code.
    
    x has df columns from location onwardsthe columns before that are from tourney games and are used to calculate y (based on points).
    
    y has label 0 or 1 (lose or win) and nan if the correspoinding data in x was nan.
    
    There is also x_final_season and y_final_season aquired from df_final, which are the same as x and y, but like df_final, they have NaNs. 
    """

    # This ensures that we simulate the scenario in competition
    tourney_results_final = tourney_results[tourney_results['Season'] == final_season]
    tourney_results = tourney_results[tourney_results['Season'] < final_season]
    regular_results = regular_results[regular_results['Season'] != 2020] # This year had no tournament data
    tourney_years_list = [[i] for i in range(start_season, final_season+1)]
    
    # Arrays to store data
    data = []
    data_final = []
    for season_years, tourney_years in zip(season_years_list, tourney_years_list):
            
        # Create data separately for the of games
        # The separation is to ensure there is no data leak
        if tourney_years[0] == final_season:
            data_tmp = get_df(
                seeds,
                regular_results, season_years, days_back,
                tourney_results_final, tourney_years,
                location_multiplier=location_multiplier,
                win_ratio_days_back=win_ratio_days_back
            )
            data_final.append(data_tmp)
        else:
            data_tmp = get_df(
                seeds,
                regular_results, season_years, days_back,
                tourney_results, tourney_years,
                location_multiplier=location_multiplier,
                win_ratio_days_back=win_ratio_days_back
            )
            data.append(data_tmp)
            
    df = pd.concat(data, ignore_index=True)
    df_final = pd.concat(data_final, ignore_index=True)
    
    return df, df_final

def brier_for_all_years(year_range, season_years_list):
    
    df_train_women_list = []
    df_test_women_list = []
    df_train_men_list = []
    df_test_men_list = []
    
    for year, season_years in zip(year_range, season_years_list):
    
        final_season = year

        for sex in ['woman', 'man']:
            
            if sex == 'woman':
                include_men = False
                include_women = True
            elif sex == 'man':
                include_men = True
                include_women = False

            # ----------------------------------------------------------
            # READ DATA
            regular_results = pd.concat([
                MRegularSeasonDetailedResults.copy() if include_men else None,
                WRegularSeasonDetailedResults.copy() if include_women else None
            ], ignore_index=True)
            tourney_results = pd.concat([
                MNCAATourneyDetailedResults.copy() if include_men else None,
                WNCAATourneyDetailedResults.copy() if include_women else None
            ], ignore_index=True)
            seeds = pd.concat([
                MNCAATourneySeeds.copy() if include_men else None,
                WNCAATourneySeeds.copy() if include_women else None
            ], ignore_index=True)
            # ----------------------------------------------------------
            # GET ALL DATA NEEDED TO USE THE MODELS
            df_train, df_test = get_all_core_data(
                regular_results, tourney_results, seeds, SampleSubmissionStage1, final_season = final_season,
                start_season = start_season, season_years_list  = season_years, days_back = days_back,
                maximum_favoured_seed = maximum_favoured_seed, location_multiplier=location_multiplier,
                include_men = include_men, include_women = include_women, win_ratio_days_back = win_ratio_days_back)
            # ----------------------------------------------------------
            # ADD TEAM AND COACH ELO AND REPLACE NAN WITH MEAN
            elo = pd.read_csv(join(data_path, 'elo.csv'))
            elo['CoachELO'] = elo['CoachELO'].fillna(elo['CoachELO'].mean())
            elo = elo.drop(['CoachName'], axis=1)
            def add_elo_column(df):
                df = df.copy()
                df = pd.merge(
                        df,
                        elo[['Season', 'DayNum', 'TeamID', 'TeamELO', 'CoachELO']],
                        left_on=['Season', 'DayNum', 'T1_TeamID'],
                        right_on=['Season', 'DayNum', 'TeamID'],
                        how='left'
                    )
                df = df.drop(['TeamID'], axis=1)
                return df
            df_train = add_elo_column(df_train)
            df_test = add_elo_column(df_test)
            # ----------------------------------------------------------
            if sex == 'woman':
                df_train_women_list.append(df_train.copy())
                df_test_women_list.append(df_test.copy())
            elif sex == 'man':
                df_train_men_list.append(df_train.copy())
                df_test_men_list.append(df_test.copy())
                
        for i in range(len(df_train_men_list)):
            df_train_women_list[i] = df_train_women_list[i][list(df_train_men_list[0])]
            df_test_women_list[i] = df_test_women_list[i][list(df_train_men_list[0])]
            
    return df_train_women_list, df_test_women_list, df_train_men_list, df_test_men_list

## Get data to use models on

In [59]:
columns_to_include_women = [
 'Season',
 'DayNum',
 'T1_TeamID',
 'T1_Score',
 'T2_TeamID',
 'T2_Score',
 'location',
                      
#  'T1_Score_mean',
 'T1_FGM',
 'T1_FGA',
 'T1_FGM3',
 'T1_FGA3',
#  'T1_FTM',
#  'T1_FTA',
 'T1_OR',
#  'T1_DR',
 'T1_Ast',
 'T1_TO',
 'T1_Stl',
#  'T1_Blk',
 'T1_PF',
                      
#  'T1_opponent_Score',
 'T1_opponent_FGM',
 'T1_opponent_FGA',
 'T1_opponent_FGM3',
 'T1_opponent_FGA3',
#  'T1_opponent_FTM',
#  'T1_opponent_FTA',
 'T1_opponent_OR',
#  'T1_opponent_DR',
 'T1_opponent_Ast',
 'T1_opponent_TO',
 'T1_opponent_Stl',
#  'T1_opponent_Blk',
 'T1_opponent_PF',
 'T1_PointDiff',
                      
#  'T2_Score_mean',
 'T2_FGM',
 'T2_FGA',
 'T2_FGM3',
 'T2_FGA3',
#  'T2_FTM',
#  'T2_FTA',
 'T2_OR',
#  'T2_DR',
 'T2_Ast',
 'T2_TO',
 'T2_Stl',
#  'T2_Blk',
 'T2_PF',
                      
#  'T2_opponent_Score',
 'T2_opponent_FGM',
 'T2_opponent_FGA',
 'T2_opponent_FGM3',
 'T2_opponent_FGA3',
#  'T2_opponent_FTM',
#  'T2_opponent_FTA',
 'T2_opponent_OR',
#  'T2_opponent_DR',
 'T2_opponent_Ast',
 'T2_opponent_TO',
 'T2_opponent_Stl',
#  'T2_opponent_Blk',
 'T2_opponent_PF',
 'T2_PointDiff',
                      
 'T1_win_ratio_14d',
 'T2_win_ratio_14d',
 'T1_seed',
 'T2_seed',
 'Seed_diff',
 'TeamELO',
#  'CoachELO'
]
columns_to_include_men = [
 'Season',
 'DayNum',
 'T1_TeamID',
 'T1_Score',
 'T2_TeamID',
 'T2_Score',
 'location',
                      
#  'T1_Score_mean',
 'T1_FGM',
 'T1_FGA',
 'T1_FGM3',
 'T1_FGA3',
#  'T1_FTM',
#  'T1_FTA',
 'T1_OR',
#  'T1_DR',
 'T1_Ast',
 'T1_TO',
 'T1_Stl',
#  'T1_Blk',
 'T1_PF',
                      
#  'T1_opponent_Score',
 'T1_opponent_FGM',
 'T1_opponent_FGA',
 'T1_opponent_FGM3',
 'T1_opponent_FGA3',
#  'T1_opponent_FTM',
#  'T1_opponent_FTA',
 'T1_opponent_OR',
#  'T1_opponent_DR',
 'T1_opponent_Ast',
 'T1_opponent_TO',
 'T1_opponent_Stl',
#  'T1_opponent_Blk',
 'T1_opponent_PF',
 'T1_PointDiff',
                      
#  'T2_Score_mean',
 'T2_FGM',
 'T2_FGA',
 'T2_FGM3',
 'T2_FGA3',
#  'T2_FTM',
#  'T2_FTA',
 'T2_OR',
#  'T2_DR',
 'T2_Ast',
 'T2_TO',
 'T2_Stl',
#  'T2_Blk',
 'T2_PF',
                      
#  'T2_opponent_Score',
 'T2_opponent_FGM',
 'T2_opponent_FGA',
 'T2_opponent_FGM3',
 'T2_opponent_FGA3',
#  'T2_opponent_FTM',
#  'T2_opponent_FTA',
 'T2_opponent_OR',
#  'T2_opponent_DR',
 'T2_opponent_Ast',
 'T2_opponent_TO',
 'T2_opponent_Stl',
#  'T2_opponent_Blk',
 'T2_opponent_PF',
 'T2_PointDiff',
                      
 'T1_win_ratio_14d',
 'T2_win_ratio_14d',
 'T1_seed',
 'T2_seed',
 'Seed_diff',
 'TeamELO',
 'CoachELO'
]

year_range = [2023]
start_season = 2003 # from which ponit should we begin creating data
season_years_list  = [[[i] for i in range(start_season, year+1)] for year in year_range] # at which seasons to look at when calculating team's stats
days_back = 150 # how many days back from the start of tourney to calculate team's stats per season
location_multiplier = [0.9, 1.1] # home penalty, away bonus ex.
win_ratio_days_back = 14 # how many days back from the tourney do we calculate win ratio
maximum_favoured_seed = 0
# -------------------------------------------
df_train_women_list, df_test_women_list, df_train_men_list, df_test_men_list = brier_for_all_years(year_range, season_years_list)
x_train_women_list = []
x_test_women_list = []
x_train_men_list = []
x_test_men_list = []
y_train_women_list = []
y_test_women_list = []
y_train_men_list = []
y_test_men_list = []
for i in range(len(year_range)):

    if len(year_range) > 1:
        df_train_women_list[i] = df_train_women_list[i][columns_to_include_women]
        df_test_women_list[i] = df_test_women_list[i][columns_to_include_women]
        df_train_men_list[i] = df_train_men_list[i][columns_to_include_men]
        df_test_men_list[i] = df_test_men_list[i][columns_to_include_men]
        
        x_train_women, y_train_women = x_y_from_data_frame(df_train_women_list[i])
        x_test_women, y_test_women = x_y_from_data_frame(df_test_women_list[i])
        x_train_men, y_train_men = x_y_from_data_frame(df_train_men_list[i])
        x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list[i])

        x_train_women, y_train_women = clear_na_from_x_y(x_train_women.copy(), y_train_women.copy())
        x_test_women, y_test_women = clear_na_from_x_y(x_test_women.copy(), y_test_women.copy())
        x_train_men, y_train_men = clear_na_from_x_y(x_train_men.copy(), y_train_men.copy())
        x_test_men, y_test_men = clear_na_from_x_y(x_test_men.copy(), y_test_men.copy())
        
        x_train_women_list.append(x_train_women)
        x_test_women_list.append(x_test_women)
        x_train_men_list.append(x_train_men)
        x_test_men_list.append(x_test_men)

        y_train_women_list.append(y_train_women)
        y_test_women_list.append(y_test_women)
        y_train_men_list.append(y_train_men)
        y_test_men_list.append(y_test_men)
    
    else:
        df_train_women_list = df_train_women_list[0][columns_to_include_women]
        df_test_women_list = df_test_women_list[0][columns_to_include_women]
        df_train_men_list = df_train_men_list[0][columns_to_include_men]
        df_test_men_list = df_test_men_list[0][columns_to_include_men]

        x_train_women, y_train_women = x_y_from_data_frame(df_train_women_list)
        x_test_women, y_test_women = x_y_from_data_frame(df_test_women_list)
        x_train_men, y_train_men = x_y_from_data_frame(df_train_men_list)
        x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list)

        x_train_women, y_train_women = clear_na_from_x_y(x_train_women.copy(), y_train_women.copy())
        x_test_women, y_test_women = clear_na_from_x_y(x_test_women.copy(), y_test_women.copy())
        x_train_men, y_train_men = clear_na_from_x_y(x_train_men.copy(), y_train_men.copy())
        x_test_men, y_test_men = clear_na_from_x_y(x_test_men.copy(), y_test_men.copy())

## Which models to consider

In [60]:
def train_and_evaluate(model, x_train, y_train, x_test, y_test):
    model.fit(x_train, y_train)
    y_pred = model.predict_proba(x_test)[:, 1]  # Get probability of class 1
    score = brier_score_loss(y_test, y_pred)
    return score

# Initialize models
catboost_model = CatBoostClassifier(verbose=0, iterations=500, depth=6, learning_rate=0.05)
xgboost_model = XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, use_label_encoder=False, eval_metric='logloss')
mlp_model = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, alpha=0.01)

# Train and evaluate for women
y_pred_women_catboost = train_and_evaluate(catboost_model, x_train_women, y_train_women, x_test_women, y_test_women)
y_pred_women_xgboost = train_and_evaluate(xgboost_model, x_train_women, y_train_women, x_test_women, y_test_women)
y_pred_women_mlp = train_and_evaluate(mlp_model, x_train_women, y_train_women, x_test_women, y_test_women)

# Train and evaluate for men
y_pred_men_catboost = train_and_evaluate(catboost_model, x_train_men, y_train_men, x_test_men, y_test_men)
y_pred_men_xgboost = train_and_evaluate(xgboost_model, x_train_men, y_train_men, x_test_men, y_test_men)
y_pred_men_mlp = train_and_evaluate(mlp_model, x_train_men, y_train_men, x_test_men, y_test_men)

# Calculate final Brier Score
final_brier_score = (y_pred_women_catboost + y_pred_women_xgboost + y_pred_women_mlp +
                     y_pred_men_catboost + y_pred_men_xgboost + y_pred_men_mlp) / 6

print("Brier Scores:")
print(f"CatBoost (Women): {y_pred_women_catboost}")
print(f"XGBoost (Women): {y_pred_women_xgboost}")
print(f"MLP (Women): {y_pred_women_mlp}")
print(f"CatBoost (Men): {y_pred_men_catboost}")
print(f"XGBoost (Men): {y_pred_men_xgboost}")
print(f"MLP (Men): {y_pred_men_mlp}")
print(f"Final Brier Score: {final_brier_score}")


Brier Scores:
CatBoost (Women): 0.18064891812105194
XGBoost (Women): 0.19179617485993633
MLP (Women): 0.23458812094220277
CatBoost (Men): 0.2046585456494985
XGBoost (Men): 0.21464189997269514
MLP (Men): 0.29452687807176575
Final Brier Score: 0.22014342293619174


# Training models

## CatBoost

In [61]:
# FOR WOMEN
def objective(trial, x_train, y_train, x_val, y_val):
    params = {
        "iterations": trial.suggest_int("iterations", 500, 3000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10, log=True),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "verbose": 0,
        "loss_function": "Logloss"
    }
    
    model = CatBoostClassifier(**params)
    model.fit(x_train, y_train, eval_set=(x_val, y_val), early_stopping_rounds=50, verbose=False)
    
    y_pred = model.predict_proba(x_val)[:, 1]
    return brier_score_loss(y_val, y_pred)

###################################################### WOMEN

# Split data into training and validation for hyperparameter tuning
x_train_women_tr, x_val_women, y_train_women_tr, y_val_women = train_test_split(x_train_women, y_train_women, test_size=0.2, random_state=42)

# Run optimization
study = optuna.create_study(direction="minimize")
study.optimize(lambda trial: objective(trial, x_train_women_tr, y_train_women_tr, x_val_women, y_val_women), n_trials=50)

# Train final model with best hyperparameters
best_params_women = study.best_params
best_catboost_women = CatBoostClassifier(**best_params_women)
best_catboost_women.fit(x_train_women, y_train_women)

# Predict on test set
y_pred_women = best_catboost_women.predict_proba(x_test_women)[:, 1]

# Compute Brier Score
brier_women = brier_score_loss(y_test_women, y_pred_women)
print("Best CatBoost Brier Score (Women):", brier_women)

###################################################### MEN

# Split data into training and validation for hyperparameter tuning
x_train_men_tr, x_val_men, y_train_men_tr, y_val_men = train_test_split(x_train_men, y_train_men, test_size=0.2, random_state=42)

# Run optimization
study = optuna.create_study(direction="minimize")
study.optimize(lambda trial: objective(trial, x_train_men_tr, y_train_men_tr, x_val_men, y_val_men), n_trials=50)

# Train final model with best hyperparameters
best_params_men = study.best_params
best_catboost_men = CatBoostClassifier(**best_params_men)
best_catboost_men.fit(x_train_men, y_train_men)

# Predict on test set
y_pred_men = best_catboost_men.predict_proba(x_test_men)[:, 1]

# Compute Brier Score
brier_men = brier_score_loss(y_test_men, y_pred_men)
print("Best CatBoost Brier Score (men):", brier_men)

[I 2025-03-19 14:06:57,205] A new study created in memory with name: no-name-4a38f382-e61a-442a-a362-6c0bc6de1bac
[I 2025-03-19 14:06:57,635] Trial 0 finished with value: 0.1423370781062303 and parameters: {'iterations': 1683, 'learning_rate': 0.10065036910619084, 'depth': 6, 'l2_leaf_reg': 0.01292535470556493, 'border_count': 86, 'random_strength': 6.542207449080143, 'bagging_temperature': 0.404074070751261}. Best is trial 0 with value: 0.1423370781062303.
[I 2025-03-19 14:06:58,923] Trial 1 finished with value: 0.14482509755747489 and parameters: {'iterations': 590, 'learning_rate': 0.12189535271401493, 'depth': 8, 'l2_leaf_reg': 0.07317937869220197, 'border_count': 171, 'random_strength': 0.0036307430322490876, 'bagging_temperature': 0.21516642317650758}. Best is trial 0 with value: 0.1423370781062303.
[I 2025-03-19 14:06:59,772] Trial 2 finished with value: 0.13442725483152781 and parameters: {'iterations': 2664, 'learning_rate': 0.046140635943929934, 'depth': 6, 'l2_leaf_reg': 0.2

[I 2025-03-19 14:07:40,181] Trial 23 finished with value: 0.13442122005332835 and parameters: {'iterations': 2985, 'learning_rate': 0.027417933566560866, 'depth': 6, 'l2_leaf_reg': 0.47333045811862706, 'border_count': 254, 'random_strength': 0.008731132334156719, 'bagging_temperature': 0.7389658249908078}. Best is trial 9 with value: 0.13062332244408942.
[I 2025-03-19 14:07:41,647] Trial 24 finished with value: 0.1339397118035146 and parameters: {'iterations': 1314, 'learning_rate': 0.017349022057548198, 'depth': 5, 'l2_leaf_reg': 2.149511372001367, 'border_count': 209, 'random_strength': 0.0024523268531926137, 'bagging_temperature': 0.5647176077288079}. Best is trial 9 with value: 0.13062332244408942.
[I 2025-03-19 14:07:43,279] Trial 25 finished with value: 0.13198830753487814 and parameters: {'iterations': 777, 'learning_rate': 0.04191524173713679, 'depth': 6, 'l2_leaf_reg': 9.20335578379462, 'border_count': 235, 'random_strength': 0.013272984658016174, 'bagging_temperature': 0.4378

[I 2025-03-19 14:08:51,529] Trial 47 finished with value: 0.13155386534377816 and parameters: {'iterations': 1883, 'learning_rate': 0.0662897258128557, 'depth': 9, 'l2_leaf_reg': 0.39157732696994624, 'border_count': 228, 'random_strength': 0.00801625750739048, 'bagging_temperature': 0.6246520576390358}. Best is trial 26 with value: 0.12912089100219173.
[I 2025-03-19 14:08:55,241] Trial 48 finished with value: 0.14210696722311789 and parameters: {'iterations': 1843, 'learning_rate': 0.0674681939150928, 'depth': 9, 'l2_leaf_reg': 0.05272485378724197, 'border_count': 255, 'random_strength': 0.0031700611097443764, 'bagging_temperature': 0.5170431258123718}. Best is trial 26 with value: 0.12912089100219173.
[I 2025-03-19 14:09:01,275] Trial 49 finished with value: 0.1421245727372132 and parameters: {'iterations': 1188, 'learning_rate': 0.17298126600236052, 'depth': 10, 'l2_leaf_reg': 0.37556217225047667, 'border_count': 187, 'random_strength': 0.045173544686080695, 'bagging_temperature': 0.

0:	learn: 0.6616539	total: 12.8ms	remaining: 10.2s
1:	learn: 0.6345782	total: 24.9ms	remaining: 9.88s
2:	learn: 0.6085588	total: 35.9ms	remaining: 9.5s
3:	learn: 0.5879754	total: 47.1ms	remaining: 9.32s
4:	learn: 0.5672477	total: 57.8ms	remaining: 9.15s
5:	learn: 0.5504486	total: 69ms	remaining: 9.09s
6:	learn: 0.5331064	total: 80.3ms	remaining: 9.05s
7:	learn: 0.5183603	total: 91ms	remaining: 8.96s
8:	learn: 0.5060349	total: 102ms	remaining: 8.91s
9:	learn: 0.4955733	total: 113ms	remaining: 8.86s
10:	learn: 0.4834006	total: 125ms	remaining: 8.9s
11:	learn: 0.4735063	total: 136ms	remaining: 8.89s
12:	learn: 0.4645201	total: 147ms	remaining: 8.88s
13:	learn: 0.4570410	total: 159ms	remaining: 8.9s
14:	learn: 0.4500166	total: 171ms	remaining: 8.89s
15:	learn: 0.4435161	total: 184ms	remaining: 8.97s
16:	learn: 0.4360175	total: 196ms	remaining: 9s
17:	learn: 0.4300098	total: 209ms	remaining: 9.04s
18:	learn: 0.4239298	total: 224ms	remaining: 9.15s
19:	learn: 0.4177839	total: 240ms	remaining

161:	learn: 0.1813155	total: 2.01s	remaining: 7.88s
162:	learn: 0.1801271	total: 2.03s	remaining: 7.87s
163:	learn: 0.1795397	total: 2.04s	remaining: 7.85s
164:	learn: 0.1787690	total: 2.05s	remaining: 7.84s
165:	learn: 0.1779097	total: 2.06s	remaining: 7.82s
166:	learn: 0.1767898	total: 2.07s	remaining: 7.8s
167:	learn: 0.1759109	total: 2.08s	remaining: 7.79s
168:	learn: 0.1758237	total: 2.09s	remaining: 7.77s
169:	learn: 0.1749677	total: 2.1s	remaining: 7.75s
170:	learn: 0.1738789	total: 2.12s	remaining: 7.73s
171:	learn: 0.1730451	total: 2.13s	remaining: 7.72s
172:	learn: 0.1720526	total: 2.14s	remaining: 7.7s
173:	learn: 0.1713157	total: 2.15s	remaining: 7.68s
174:	learn: 0.1712214	total: 2.16s	remaining: 7.67s
175:	learn: 0.1700629	total: 2.17s	remaining: 7.65s
176:	learn: 0.1693387	total: 2.18s	remaining: 7.63s
177:	learn: 0.1692159	total: 2.19s	remaining: 7.61s
178:	learn: 0.1685560	total: 2.2s	remaining: 7.6s
179:	learn: 0.1675558	total: 2.22s	remaining: 7.59s
180:	learn: 0.166

...

1996:	learn: 0.0399396	total: 28.1s	remaining: 10.6s
1997:	learn: 0.0399001	total: 28.1s	remaining: 10.6s
1998:	learn: 0.0398624	total: 28.1s	remaining: 10.6s
1999:	learn: 0.0398231	total: 28.1s	remaining: 10.6s
2000:	learn: 0.0397871	total: 28.2s	remaining: 10.5s
2001:	learn: 0.0397233	total: 28.2s	remaining: 10.5s
2002:	learn: 0.0396972	total: 28.2s	remaining: 10.5s
2003:	learn: 0.0396716	total: 28.2s	remaining: 10.5s
2004:	learn: 0.0396203	total: 28.2s	remaining: 10.5s
2005:	learn: 0.0395625	total: 28.2s	remaining: 10.5s
2006:	learn: 0.0395053	total: 28.2s	remaining: 10.5s
2007:	learn: 0.0394699	total: 28.3s	remaining: 10.4s
2008:	learn: 0.0394321	total: 28.3s	remaining: 10.4s
2009:	learn: 0.0393726	total: 28.3s	remaining: 10.4s
2010:	learn: 0.0393280	total: 28.3s	remaining: 10.4s
2011:	learn: 0.0392877	total: 28.3s	remaining: 10.4s
2012:	learn: 0.0392574	total: 28.3s	remaining: 10.4s
2013:	learn: 0.0392006	total: 28.3s	remaining: 10.4s
2014:	learn: 0.0391548	total: 28.4s	remaining:

2164:	learn: 0.0332928	total: 30.4s	remaining: 8.21s
2165:	learn: 0.0332607	total: 30.4s	remaining: 8.2s
2166:	learn: 0.0332240	total: 30.4s	remaining: 8.19s
2167:	learn: 0.0331829	total: 30.4s	remaining: 8.17s
2168:	learn: 0.0331490	total: 30.5s	remaining: 8.16s
2169:	learn: 0.0331232	total: 30.5s	remaining: 8.14s
2170:	learn: 0.0330705	total: 30.5s	remaining: 8.13s
2171:	learn: 0.0330510	total: 30.5s	remaining: 8.12s
2172:	learn: 0.0329942	total: 30.5s	remaining: 8.1s
2173:	learn: 0.0329520	total: 30.5s	remaining: 8.09s
2174:	learn: 0.0329248	total: 30.5s	remaining: 8.07s
2175:	learn: 0.0328896	total: 30.5s	remaining: 8.06s
2176:	learn: 0.0328580	total: 30.6s	remaining: 8.04s
2177:	learn: 0.0328314	total: 30.6s	remaining: 8.03s
2178:	learn: 0.0328146	total: 30.6s	remaining: 8.02s
2179:	learn: 0.0327956	total: 30.6s	remaining: 8s
2180:	learn: 0.0327609	total: 30.6s	remaining: 7.99s
2181:	learn: 0.0327358	total: 30.6s	remaining: 7.97s
2182:	learn: 0.0327019	total: 30.6s	remaining: 7.96

2326:	learn: 0.0281857	total: 32.7s	remaining: 5.94s
2327:	learn: 0.0281671	total: 32.7s	remaining: 5.93s
2328:	learn: 0.0281453	total: 32.8s	remaining: 5.92s
2329:	learn: 0.0281242	total: 32.8s	remaining: 5.91s
2330:	learn: 0.0281067	total: 32.8s	remaining: 5.89s
2331:	learn: 0.0280778	total: 32.8s	remaining: 5.88s
2332:	learn: 0.0280562	total: 32.8s	remaining: 5.86s
2333:	learn: 0.0280309	total: 32.8s	remaining: 5.85s
2334:	learn: 0.0279965	total: 32.8s	remaining: 5.83s
2335:	learn: 0.0279618	total: 32.8s	remaining: 5.82s
2336:	learn: 0.0279355	total: 32.9s	remaining: 5.81s
2337:	learn: 0.0279119	total: 32.9s	remaining: 5.79s
2338:	learn: 0.0278618	total: 32.9s	remaining: 5.78s
2339:	learn: 0.0278370	total: 32.9s	remaining: 5.76s
2340:	learn: 0.0278187	total: 32.9s	remaining: 5.75s
2341:	learn: 0.0278005	total: 32.9s	remaining: 5.74s
2342:	learn: 0.0277751	total: 32.9s	remaining: 5.72s
2343:	learn: 0.0277607	total: 33s	remaining: 5.71s
2344:	learn: 0.0277436	total: 33s	remaining: 5.6

2489:	learn: 0.0240145	total: 34.9s	remaining: 3.65s
2490:	learn: 0.0239876	total: 35s	remaining: 3.63s
2491:	learn: 0.0239653	total: 35s	remaining: 3.62s
2492:	learn: 0.0239409	total: 35s	remaining: 3.61s
2493:	learn: 0.0239180	total: 35s	remaining: 3.59s
2494:	learn: 0.0238909	total: 35s	remaining: 3.58s
2495:	learn: 0.0238724	total: 35s	remaining: 3.56s
2496:	learn: 0.0238562	total: 35s	remaining: 3.55s
2497:	learn: 0.0238349	total: 35.1s	remaining: 3.54s
2498:	learn: 0.0238136	total: 35.1s	remaining: 3.52s
2499:	learn: 0.0237930	total: 35.1s	remaining: 3.51s
2500:	learn: 0.0237617	total: 35.1s	remaining: 3.49s
2501:	learn: 0.0237427	total: 35.1s	remaining: 3.48s
2502:	learn: 0.0237181	total: 35.1s	remaining: 3.46s
2503:	learn: 0.0237039	total: 35.1s	remaining: 3.45s
2504:	learn: 0.0236863	total: 35.1s	remaining: 3.44s
2505:	learn: 0.0236712	total: 35.2s	remaining: 3.42s
2506:	learn: 0.0236474	total: 35.2s	remaining: 3.41s
2507:	learn: 0.0236252	total: 35.2s	remaining: 3.4s
2508:	le

2658:	learn: 0.0205278	total: 37.2s	remaining: 1.27s
2659:	learn: 0.0204969	total: 37.2s	remaining: 1.26s
2660:	learn: 0.0204787	total: 37.3s	remaining: 1.25s
2661:	learn: 0.0204563	total: 37.3s	remaining: 1.23s
2662:	learn: 0.0204381	total: 37.3s	remaining: 1.22s
2663:	learn: 0.0204112	total: 37.3s	remaining: 1.2s
2664:	learn: 0.0203966	total: 37.3s	remaining: 1.19s
2665:	learn: 0.0203814	total: 37.3s	remaining: 1.18s
2666:	learn: 0.0203654	total: 37.3s	remaining: 1.16s
2667:	learn: 0.0203480	total: 37.4s	remaining: 1.15s
2668:	learn: 0.0203308	total: 37.4s	remaining: 1.13s
2669:	learn: 0.0203129	total: 37.4s	remaining: 1.12s
2670:	learn: 0.0202905	total: 37.4s	remaining: 1.1s
2671:	learn: 0.0202592	total: 37.4s	remaining: 1.09s
2672:	learn: 0.0202423	total: 37.4s	remaining: 1.08s
2673:	learn: 0.0202261	total: 37.4s	remaining: 1.06s
2674:	learn: 0.0202122	total: 37.4s	remaining: 1.05s
2675:	learn: 0.0201987	total: 37.5s	remaining: 1.03s
2676:	learn: 0.0201739	total: 37.5s	remaining: 1

## Logistic regression

In [62]:
def objective_lr(trial, x_train, y_train, x_val, y_val):
    C = trial.suggest_loguniform("C", 1e-4, 10)
    
    # Define the model pipeline with scaling
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(C=C, solver="liblinear", max_iter=1000))
    ])
    
    # Train the model
    model.fit(x_train, y_train)
    
    # Predict probabilities
    y_pred = model.predict_proba(x_val)[:, 1]
    
    # Return Brier Score as the optimization metric
    return brier_score_loss(y_val, y_pred)


###################################################### WOMEN

# Split data for tuning
x_train_women_tr, x_val_women, y_train_women_tr, y_val_women = train_test_split(
    x_train_women, y_train_women, test_size=0.2, random_state=42
)

# Run Optuna optimization
study = optuna.create_study(direction="minimize")
study.optimize(lambda trial: objective_lr(trial, x_train_women_tr, y_train_women_tr, x_val_women, y_val_women), n_trials=50)

# Get best parameters
best_params_women_lr = study.best_params
print("Best Logistic Regression Params (Women):", best_params_women_lr)

# Train final Logistic Regression model with best params
best_lr_women = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(C=best_params_women_lr["C"], solver="liblinear", max_iter=1000))
])
best_lr_women.fit(x_train_women, y_train_women)

# Predict on test set
y_pred_women_lr = best_lr_women.predict_proba(x_test_women)[:, 1]

# Compute Brier Score
brier_women_lr = brier_score_loss(y_test_women, y_pred_women_lr)
print("Best Logistic Regression Brier Score (Women):", brier_women_lr)


###################################################### MEN

# Split data for tuning
x_train_men_tr, x_val_men, y_train_men_tr, y_val_men = train_test_split(
    x_train_men, y_train_men, test_size=0.2, random_state=42
)

# Run Optuna optimization
study = optuna.create_study(direction="minimize")
study.optimize(lambda trial: objective_lr(trial, x_train_men_tr, y_train_men_tr, x_val_men, y_val_men), n_trials=50)

# Get best parameters
best_params_men_lr = study.best_params
print("Best Logistic Regression Params (Men):", best_params_men_lr)

# Train final Logistic Regression model with best params
best_lr_men = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(C=best_params_men_lr["C"], solver="liblinear", max_iter=1000))
])
best_lr_men.fit(x_train_men, y_train_men)

# Predict on test set
y_pred_men_lr = best_lr_men.predict_proba(x_test_men)[:, 1]

# Compute Brier Score
brier_men_lr = brier_score_loss(y_test_men, y_pred_men_lr)
print("Best Logistic Regression Brier Score (Men):", brier_men_lr)

[I 2025-03-19 14:12:59,169] A new study created in memory with name: no-name-ff6e8e30-8e0d-4640-bca1-5e5bf3f53889
C:\Users\Sebastian\AppData\Local\Temp\ipykernel_7980\2391107513.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform("C", 1e-4, 10)
[I 2025-03-19 14:12:59,187] Trial 0 finished with value: 0.13741225197198004 and parameters: {'C': 0.044916310104202335}. Best is trial 0 with value: 0.13741225197198004.
C:\Users\Sebastian\AppData\Local\Temp\ipykernel_7980\2391107513.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform("C", 1e-4, 10)
[I 2025-03-19 14:12:59,206] Trial 1 finished with value: 0.13999995199842

C:\Users\Sebastian\AppData\Local\Temp\ipykernel_7980\2391107513.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform("C", 1e-4, 10)
[I 2025-03-19 14:12:59,551] Trial 17 finished with value: 0.13670321886753892 and parameters: {'C': 0.015676299788495645}. Best is trial 17 with value: 0.13670321886753892.
C:\Users\Sebastian\AppData\Local\Temp\ipykernel_7980\2391107513.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform("C", 1e-4, 10)
[I 2025-03-19 14:12:59,589] Trial 18 finished with value: 0.168985813955662 and parameters: {'C': 0.0006918001920633279}. Best is trial 17 with value: 0.13670321886753892.
C:\Users\Seba

[I 2025-03-19 14:12:59,906] Trial 33 finished with value: 0.1390222448288892 and parameters: {'C': 0.005070426274475636}. Best is trial 17 with value: 0.13670321886753892.
C:\Users\Sebastian\AppData\Local\Temp\ipykernel_7980\2391107513.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform("C", 1e-4, 10)
[I 2025-03-19 14:12:59,927] Trial 34 finished with value: 0.13676525972710896 and parameters: {'C': 0.012628984802912949}. Best is trial 17 with value: 0.13670321886753892.
C:\Users\Sebastian\AppData\Local\Temp\ipykernel_7980\2391107513.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform("C", 1e-4, 10)
[I 2025-03-19

[I 2025-03-19 14:13:00,364] Trial 0 finished with value: 0.17711433611831479 and parameters: {'C': 0.22956901537528254}. Best is trial 0 with value: 0.17711433611831479.
C:\Users\Sebastian\AppData\Local\Temp\ipykernel_7980\2391107513.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform("C", 1e-4, 10)
[I 2025-03-19 14:13:00,380] Trial 1 finished with value: 0.21686602395316706 and parameters: {'C': 0.00018598522625983905}. Best is trial 0 with value: 0.17711433611831479.
C:\Users\Sebastian\AppData\Local\Temp\ipykernel_7980\2391107513.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform("C", 1e-4, 10)
[I 2025-03-19 1

Best Logistic Regression Params (Women): {'C': 0.015676299788495645}
Best Logistic Regression Brier Score (Women): 0.1620919391783939


C:\Users\Sebastian\AppData\Local\Temp\ipykernel_7980\2391107513.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform("C", 1e-4, 10)
[I 2025-03-19 14:13:00,525] Trial 9 finished with value: 0.17789676365220794 and parameters: {'C': 1.410470068722511}. Best is trial 5 with value: 0.17654059549865278.
C:\Users\Sebastian\AppData\Local\Temp\ipykernel_7980\2391107513.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform("C", 1e-4, 10)
[I 2025-03-19 14:13:00,554] Trial 10 finished with value: 0.1766421403830554 and parameters: {'C': 0.085647160032856}. Best is trial 5 with value: 0.17654059549865278.
C:\Users\Sebastian\App

C:\Users\Sebastian\AppData\Local\Temp\ipykernel_7980\2391107513.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform("C", 1e-4, 10)
[I 2025-03-19 14:13:01,028] Trial 26 finished with value: 0.1765337376561179 and parameters: {'C': 0.03690586788883747}. Best is trial 17 with value: 0.17652929852855254.
C:\Users\Sebastian\AppData\Local\Temp\ipykernel_7980\2391107513.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform("C", 1e-4, 10)
[I 2025-03-19 14:13:01,043] Trial 27 finished with value: 0.17761956898601772 and parameters: {'C': 0.008367557522583379}. Best is trial 17 with value: 0.17652929852855254.
C:\Users\Sebas

C:\Users\Sebastian\AppData\Local\Temp\ipykernel_7980\2391107513.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform("C", 1e-4, 10)
[I 2025-03-19 14:13:01,459] Trial 43 finished with value: 0.1767076864315266 and parameters: {'C': 0.02085334527962522}. Best is trial 33 with value: 0.17652461790460058.
C:\Users\Sebastian\AppData\Local\Temp\ipykernel_7980\2391107513.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform("C", 1e-4, 10)
[I 2025-03-19 14:13:01,472] Trial 44 finished with value: 0.17720909746673466 and parameters: {'C': 0.011268473775706602}. Best is trial 33 with value: 0.17652461790460058.
C:\Users\Sebas

Best Logistic Regression Params (Men): {'C': 0.04339501968402461}
Best Logistic Regression Brier Score (Men): 0.2077630730127668


# Evaluate

In [77]:
def evaluate(model, x_train, y_train, x_test, y_test):
    model.fit(x_train, y_train)
    y_pred = model.predict_proba(x_test)[:, 1]  # Get probability of class 1
    score = brier_score_loss(y_test, y_pred)
    return score

# Initialize models
# catboost_model = CatBoostClassifier(**be)
# xgboost_model = XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, use_label_encoder=False, eval_metric='logloss')
# mlp_model = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, alpha=0.01)
# lr_model = Pipeline([
#     ("scaler", StandardScaler()),
#     ("classifier", LogisticRegression(C=best_params_men_lr["C"], solver="liblinear", max_iter=1000))
# ])

# Train and evaluate for women
y_pred_women_catboost = evaluate(CatBoostClassifier(**best_params_women), x_train_women, y_train_women, x_test_women, y_test_women)
y_pred_men_lr = evaluate(Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(C=best_params_men_lr["C"], solver="liblinear", max_iter=1000))
]), x_train_men, y_train_men, x_test_men, y_test_men)

# Train and evaluate for men
y_pred_men_catboost = evaluate(CatBoostClassifier(**best_params_men), x_train_men, y_train_men, x_test_men, y_test_men)
y_pred_women_lr = evaluate(Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(C=best_params_women_lr["C"], solver="liblinear", max_iter=1000))
]), x_train_women, y_train_women, x_test_women, y_test_women)

# Calculate final Brier Score
# final_brier_score = (y_pred_women_catboost + y_pred_women_xgboost + y_pred_women_mlp +
#                      y_pred_men_catboost + y_pred_men_xgboost + y_pred_men_mlp) / 6

print("Brier Scores:")
print(f"CatBoost (Women): {y_pred_women_catboost}")
print(f"Logistic (Women): {y_pred_women_lr}")
print(f"CatBoost (Men): {y_pred_men_catboost}")
print(f"Logistic (men): {y_pred_men_lr}")
# print(f"Final Brier Score: {final_brier_score}")


0:	learn: 0.6616539	total: 29ms	remaining: 23.1s
1:	learn: 0.6345782	total: 39ms	remaining: 15.5s
2:	learn: 0.6085588	total: 49.1ms	remaining: 13s
3:	learn: 0.5879754	total: 58.7ms	remaining: 11.6s
4:	learn: 0.5672477	total: 68.3ms	remaining: 10.8s
5:	learn: 0.5504486	total: 78.2ms	remaining: 10.3s
6:	learn: 0.5331064	total: 87.6ms	remaining: 9.87s
7:	learn: 0.5183603	total: 97.3ms	remaining: 9.59s
8:	learn: 0.5060349	total: 107ms	remaining: 9.37s
9:	learn: 0.4955733	total: 117ms	remaining: 9.21s
10:	learn: 0.4834006	total: 127ms	remaining: 9.04s
11:	learn: 0.4735063	total: 136ms	remaining: 8.88s
12:	learn: 0.4645201	total: 145ms	remaining: 8.73s
13:	learn: 0.4570410	total: 154ms	remaining: 8.61s
14:	learn: 0.4500166	total: 163ms	remaining: 8.5s
15:	learn: 0.4435161	total: 172ms	remaining: 8.4s
16:	learn: 0.4360175	total: 181ms	remaining: 8.32s
17:	learn: 0.4300098	total: 190ms	remaining: 8.23s
18:	learn: 0.4239298	total: 200ms	remaining: 8.17s
19:	learn: 0.4177839	total: 209ms	remaini

168:	learn: 0.1758237	total: 1.82s	remaining: 6.74s
169:	learn: 0.1749677	total: 1.83s	remaining: 6.74s
170:	learn: 0.1738789	total: 1.84s	remaining: 6.73s
171:	learn: 0.1730451	total: 1.85s	remaining: 6.72s
172:	learn: 0.1720526	total: 1.86s	remaining: 6.71s
173:	learn: 0.1713157	total: 1.88s	remaining: 6.71s
174:	learn: 0.1712214	total: 1.89s	remaining: 6.7s
175:	learn: 0.1700629	total: 1.9s	remaining: 6.69s
176:	learn: 0.1693387	total: 1.91s	remaining: 6.68s
177:	learn: 0.1692159	total: 1.92s	remaining: 6.67s
178:	learn: 0.1685560	total: 1.93s	remaining: 6.66s
179:	learn: 0.1675558	total: 1.94s	remaining: 6.65s
180:	learn: 0.1667709	total: 1.95s	remaining: 6.64s
181:	learn: 0.1660487	total: 1.97s	remaining: 6.63s
182:	learn: 0.1654529	total: 1.98s	remaining: 6.62s
183:	learn: 0.1643344	total: 1.99s	remaining: 6.61s
184:	learn: 0.1636775	total: 2s	remaining: 6.6s
185:	learn: 0.1628157	total: 2.01s	remaining: 6.6s
186:	learn: 0.1618791	total: 2.03s	remaining: 6.6s
187:	learn: 0.160670

330:	learn: 0.0930260	total: 4.01s	remaining: 5.63s
331:	learn: 0.0928193	total: 4.02s	remaining: 5.62s
332:	learn: 0.0924293	total: 4.03s	remaining: 5.61s
333:	learn: 0.0920296	total: 4.05s	remaining: 5.6s
334:	learn: 0.0917809	total: 4.06s	remaining: 5.59s
335:	learn: 0.0917316	total: 4.08s	remaining: 5.58s
336:	learn: 0.0916496	total: 4.09s	remaining: 5.57s
337:	learn: 0.0915553	total: 4.1s	remaining: 5.56s
338:	learn: 0.0911952	total: 4.12s	remaining: 5.55s
339:	learn: 0.0908294	total: 4.13s	remaining: 5.54s
340:	learn: 0.0905648	total: 4.14s	remaining: 5.53s
341:	learn: 0.0901990	total: 4.16s	remaining: 5.52s
342:	learn: 0.0901585	total: 4.17s	remaining: 5.51s
343:	learn: 0.0898391	total: 4.19s	remaining: 5.5s
344:	learn: 0.0896938	total: 4.2s	remaining: 5.49s
345:	learn: 0.0893813	total: 4.21s	remaining: 5.48s
346:	learn: 0.0892318	total: 4.23s	remaining: 5.47s
347:	learn: 0.0888353	total: 4.24s	remaining: 5.46s
348:	learn: 0.0884795	total: 4.26s	remaining: 5.45s
349:	learn: 0.08

491:	learn: 0.0567435	total: 6.19s	remaining: 3.82s
492:	learn: 0.0566424	total: 6.2s	remaining: 3.81s
493:	learn: 0.0564391	total: 6.21s	remaining: 3.8s
494:	learn: 0.0562927	total: 6.23s	remaining: 3.79s
495:	learn: 0.0562853	total: 6.24s	remaining: 3.77s
496:	learn: 0.0562629	total: 6.25s	remaining: 3.76s
497:	learn: 0.0560992	total: 6.27s	remaining: 3.75s
498:	learn: 0.0558667	total: 6.28s	remaining: 3.74s
499:	learn: 0.0556800	total: 6.3s	remaining: 3.73s
500:	learn: 0.0555126	total: 6.31s	remaining: 3.72s
501:	learn: 0.0553437	total: 6.33s	remaining: 3.71s
502:	learn: 0.0552668	total: 6.34s	remaining: 3.69s
503:	learn: 0.0551506	total: 6.35s	remaining: 3.68s
504:	learn: 0.0549701	total: 6.37s	remaining: 3.67s
505:	learn: 0.0547975	total: 6.38s	remaining: 3.66s
506:	learn: 0.0546207	total: 6.4s	remaining: 3.65s
507:	learn: 0.0545137	total: 6.41s	remaining: 3.64s
508:	learn: 0.0543021	total: 6.43s	remaining: 3.62s
509:	learn: 0.0541676	total: 6.44s	remaining: 3.61s
510:	learn: 0.05

662:	learn: 0.0391277	total: 8.4s	remaining: 1.68s
663:	learn: 0.0390208	total: 8.41s	remaining: 1.67s
664:	learn: 0.0389267	total: 8.43s	remaining: 1.66s
665:	learn: 0.0388094	total: 8.44s	remaining: 1.65s
666:	learn: 0.0387672	total: 8.46s	remaining: 1.64s
667:	learn: 0.0386571	total: 8.47s	remaining: 1.62s
668:	learn: 0.0385485	total: 8.49s	remaining: 1.61s
669:	learn: 0.0384472	total: 8.5s	remaining: 1.6s
670:	learn: 0.0384048	total: 8.51s	remaining: 1.58s
671:	learn: 0.0383036	total: 8.53s	remaining: 1.57s
672:	learn: 0.0382989	total: 8.54s	remaining: 1.56s
673:	learn: 0.0382355	total: 8.55s	remaining: 1.55s
674:	learn: 0.0381705	total: 8.57s	remaining: 1.54s
675:	learn: 0.0380462	total: 8.58s	remaining: 1.52s
676:	learn: 0.0379470	total: 8.6s	remaining: 1.51s
677:	learn: 0.0378640	total: 8.61s	remaining: 1.5s
678:	learn: 0.0378014	total: 8.63s	remaining: 1.49s
679:	learn: 0.0377001	total: 8.64s	remaining: 1.47s
680:	learn: 0.0375553	total: 8.66s	remaining: 1.46s
681:	learn: 0.037

...

2032:	learn: 0.0383639	total: 29.6s	remaining: 10.4s
2033:	learn: 0.0383069	total: 29.6s	remaining: 10.4s
2034:	learn: 0.0382735	total: 29.6s	remaining: 10.4s
2035:	learn: 0.0382150	total: 29.7s	remaining: 10.4s
2036:	learn: 0.0381772	total: 29.7s	remaining: 10.4s
2037:	learn: 0.0381253	total: 29.7s	remaining: 10.4s
2038:	learn: 0.0380779	total: 29.7s	remaining: 10.4s
2039:	learn: 0.0380221	total: 29.7s	remaining: 10.3s
2040:	learn: 0.0379884	total: 29.7s	remaining: 10.3s
2041:	learn: 0.0379283	total: 29.7s	remaining: 10.3s
2042:	learn: 0.0378981	total: 29.8s	remaining: 10.3s
2043:	learn: 0.0378343	total: 29.8s	remaining: 10.3s
2044:	learn: 0.0377933	total: 29.8s	remaining: 10.3s
2045:	learn: 0.0377566	total: 29.8s	remaining: 10.3s
2046:	learn: 0.0377092	total: 29.8s	remaining: 10.2s
2047:	learn: 0.0376454	total: 29.8s	remaining: 10.2s
2048:	learn: 0.0375966	total: 29.8s	remaining: 10.2s
2049:	learn: 0.0375510	total: 29.9s	remaining: 10.2s
2050:	learn: 0.0375080	total: 29.9s	remaining:

2197:	learn: 0.0321248	total: 31.9s	remaining: 8.01s
2198:	learn: 0.0320999	total: 31.9s	remaining: 8s
2199:	learn: 0.0320802	total: 31.9s	remaining: 7.98s
2200:	learn: 0.0320493	total: 31.9s	remaining: 7.97s
2201:	learn: 0.0320142	total: 32s	remaining: 7.95s
2202:	learn: 0.0319806	total: 32s	remaining: 7.94s
2203:	learn: 0.0319455	total: 32s	remaining: 7.92s
2204:	learn: 0.0319302	total: 32s	remaining: 7.91s
2205:	learn: 0.0319046	total: 32s	remaining: 7.89s
2206:	learn: 0.0318696	total: 32s	remaining: 7.88s
2207:	learn: 0.0318468	total: 32s	remaining: 7.86s
2208:	learn: 0.0318154	total: 32.1s	remaining: 7.85s
2209:	learn: 0.0317977	total: 32.1s	remaining: 7.83s
2210:	learn: 0.0317639	total: 32.1s	remaining: 7.82s
2211:	learn: 0.0317318	total: 32.1s	remaining: 7.8s
2212:	learn: 0.0316898	total: 32.1s	remaining: 7.79s
2213:	learn: 0.0316590	total: 32.1s	remaining: 7.78s
2214:	learn: 0.0316284	total: 32.1s	remaining: 7.76s
2215:	learn: 0.0315984	total: 32.1s	remaining: 7.75s
2216:	learn

2363:	learn: 0.0272731	total: 34.2s	remaining: 5.58s
2364:	learn: 0.0272328	total: 34.2s	remaining: 5.57s
2365:	learn: 0.0271947	total: 34.2s	remaining: 5.55s
2366:	learn: 0.0271776	total: 34.2s	remaining: 5.54s
2367:	learn: 0.0271557	total: 34.2s	remaining: 5.52s
2368:	learn: 0.0271316	total: 34.2s	remaining: 5.51s
2369:	learn: 0.0270984	total: 34.3s	remaining: 5.49s
2370:	learn: 0.0270762	total: 34.3s	remaining: 5.48s
2371:	learn: 0.0270375	total: 34.3s	remaining: 5.46s
2372:	learn: 0.0270137	total: 34.3s	remaining: 5.45s
2373:	learn: 0.0269699	total: 34.3s	remaining: 5.43s
2374:	learn: 0.0269436	total: 34.3s	remaining: 5.42s
2375:	learn: 0.0269111	total: 34.3s	remaining: 5.4s
2376:	learn: 0.0268771	total: 34.3s	remaining: 5.39s
2377:	learn: 0.0268250	total: 34.4s	remaining: 5.37s
2378:	learn: 0.0268017	total: 34.4s	remaining: 5.36s
2379:	learn: 0.0267786	total: 34.4s	remaining: 5.35s
2380:	learn: 0.0267593	total: 34.4s	remaining: 5.33s
2381:	learn: 0.0267308	total: 34.4s	remaining: 

2532:	learn: 0.0231028	total: 36.5s	remaining: 3.12s
2533:	learn: 0.0230796	total: 36.5s	remaining: 3.11s
2534:	learn: 0.0230546	total: 36.5s	remaining: 3.09s
2535:	learn: 0.0230306	total: 36.5s	remaining: 3.08s
2536:	learn: 0.0230126	total: 36.5s	remaining: 3.06s
2537:	learn: 0.0229952	total: 36.5s	remaining: 3.05s
2538:	learn: 0.0229769	total: 36.5s	remaining: 3.04s
2539:	learn: 0.0229642	total: 36.6s	remaining: 3.02s
2540:	learn: 0.0229445	total: 36.6s	remaining: 3.01s
2541:	learn: 0.0229160	total: 36.6s	remaining: 2.99s
2542:	learn: 0.0228951	total: 36.6s	remaining: 2.98s
2543:	learn: 0.0228640	total: 36.6s	remaining: 2.96s
2544:	learn: 0.0228459	total: 36.6s	remaining: 2.95s
2545:	learn: 0.0228243	total: 36.6s	remaining: 2.94s
2546:	learn: 0.0227974	total: 36.6s	remaining: 2.92s
2547:	learn: 0.0227722	total: 36.7s	remaining: 2.91s
2548:	learn: 0.0227529	total: 36.7s	remaining: 2.89s
2549:	learn: 0.0227334	total: 36.7s	remaining: 2.88s
2550:	learn: 0.0227142	total: 36.7s	remaining:

2699:	learn: 0.0197211	total: 38.7s	remaining: 717ms
2700:	learn: 0.0196952	total: 38.8s	remaining: 703ms
2701:	learn: 0.0196870	total: 38.8s	remaining: 689ms
2702:	learn: 0.0196800	total: 38.8s	remaining: 674ms
2703:	learn: 0.0196616	total: 38.8s	remaining: 660ms
2704:	learn: 0.0196450	total: 38.8s	remaining: 646ms
2705:	learn: 0.0196315	total: 38.8s	remaining: 631ms
2706:	learn: 0.0196211	total: 38.8s	remaining: 617ms
2707:	learn: 0.0196013	total: 38.9s	remaining: 603ms
2708:	learn: 0.0195896	total: 38.9s	remaining: 588ms
2709:	learn: 0.0195683	total: 38.9s	remaining: 574ms
2710:	learn: 0.0195539	total: 38.9s	remaining: 559ms
2711:	learn: 0.0195399	total: 38.9s	remaining: 545ms
2712:	learn: 0.0195191	total: 38.9s	remaining: 531ms
2713:	learn: 0.0195000	total: 38.9s	remaining: 516ms
2714:	learn: 0.0194772	total: 38.9s	remaining: 502ms
2715:	learn: 0.0194588	total: 39s	remaining: 488ms
2716:	learn: 0.0194411	total: 39s	remaining: 473ms
2717:	learn: 0.0194166	total: 39s	remaining: 459ms

### Training and testing CatBoost

In [21]:
# FOR WOMEN
def objective(trial, x_train, y_train, x_val, y_val):
    params = {
        "iterations": trial.suggest_int("iterations", 500, 3000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10, log=True),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "verbose": 0,
        "loss_function": "Logloss"
    }
    
    model = CatBoostClassifier(**params)
    model.fit(x_train, y_train, eval_set=(x_val, y_val), early_stopping_rounds=50, verbose=False)
    
    y_pred = model.predict_proba(x_val)[:, 1]
    return brier_score_loss(y_val, y_pred)

###################################################### WOMEN

# Split data into training and validation for hyperparameter tuning
x_train_women_tr, x_val_women, y_train_women_tr, y_val_women = train_test_split(x_train_women, y_train_women, test_size=0.2, random_state=42)

# Run optimization
study = optuna.create_study(direction="minimize")
study.optimize(lambda trial: objective(trial, x_train_women_tr, y_train_women_tr, x_val_women, y_val_women), n_trials=50)

# Train final model with best hyperparameters
best_params_women = study.best_params
best_catboost_women = CatBoostClassifier(**best_params_women)
best_catboost_women.fit(x_train_women, y_train_women)

# Predict on test set
y_pred_women = best_catboost_women.predict_proba(x_test_women)[:, 1]

# Compute Brier Score
brier_women = brier_score_loss(y_test_women, y_pred_women)
print("Best CatBoost Brier Score (Women):", brier_women)

###################################################### MEN

# Split data into training and validation for hyperparameter tuning
x_train_men_tr, x_val_men, y_train_men_tr, y_val_men = train_test_split(x_train_men, y_train_men, test_size=0.2, random_state=42)

# Run optimization
study = optuna.create_study(direction="minimize")
study.optimize(lambda trial: objective(trial, x_train_men_tr, y_train_men_tr, x_val_men, y_val_men), n_trials=50)

# Train final model with best hyperparameters
best_params_men = study.best_params
best_catboost_men = CatBoostClassifier(**best_params_men)
best_catboost_men.fit(x_train_men, y_train_men)

# Predict on test set
y_pred_men = best_catboost_men.predict_proba(x_test_men)[:, 1]

# Compute Brier Score
brier_men = brier_score_loss(y_test_men, y_pred_men)
print("Best CatBoost Brier Score (men):", brier_men)

[I 2025-03-19 12:51:24,908] A new study created in memory with name: no-name-9c65cb18-d93a-47d6-a3dd-588c89ca51fe
[I 2025-03-19 12:51:29,184] Trial 0 finished with value: 0.1346665596910802 and parameters: {'iterations': 1644, 'learning_rate': 0.01460245576249046, 'depth': 5, 'l2_leaf_reg': 0.12578445855475587, 'border_count': 192, 'random_strength': 0.9786739674383382, 'bagging_temperature': 0.908904405243431}. Best is trial 0 with value: 0.1346665596910802.
[I 2025-03-19 12:51:34,967] Trial 1 finished with value: 0.13687309785641563 and parameters: {'iterations': 1520, 'learning_rate': 0.034751981557127797, 'depth': 9, 'l2_leaf_reg': 8.408136712880681, 'border_count': 92, 'random_strength': 0.223800385134543, 'bagging_temperature': 0.4219472444002772}. Best is trial 0 with value: 0.1346665596910802.
[I 2025-03-19 12:51:36,965] Trial 2 finished with value: 0.1338745029278254 and parameters: {'iterations': 703, 'learning_rate': 0.026006359237062994, 'depth': 6, 'l2_leaf_reg': 0.2423871

[I 2025-03-19 12:52:17,829] Trial 23 finished with value: 0.13439515876117186 and parameters: {'iterations': 1371, 'learning_rate': 0.024668893353134432, 'depth': 5, 'l2_leaf_reg': 0.29679725859457645, 'border_count': 157, 'random_strength': 0.007514035249015495, 'bagging_temperature': 0.3360356494930174}. Best is trial 21 with value: 0.1326452523801435.
[I 2025-03-19 12:52:18,554] Trial 24 finished with value: 0.1363461950340043 and parameters: {'iterations': 729, 'learning_rate': 0.046549231517348334, 'depth': 6, 'l2_leaf_reg': 0.05067800814644887, 'border_count': 70, 'random_strength': 0.0019880217295973877, 'bagging_temperature': 0.5450686323951258}. Best is trial 21 with value: 0.1326452523801435.
[I 2025-03-19 12:52:19,634] Trial 25 finished with value: 0.13557094146335466 and parameters: {'iterations': 2129, 'learning_rate': 0.017399204416807607, 'depth': 4, 'l2_leaf_reg': 0.8694223959831106, 'border_count': 117, 'random_strength': 0.0190759309759865, 'bagging_temperature': 0.73

[I 2025-03-19 12:52:57,949] Trial 47 finished with value: 0.13680902507316287 and parameters: {'iterations': 748, 'learning_rate': 0.09423373262710621, 'depth': 4, 'l2_leaf_reg': 0.0010648486590155895, 'border_count': 228, 'random_strength': 0.011121265471942454, 'bagging_temperature': 0.305455298630254}. Best is trial 30 with value: 0.13078553985762809.
[I 2025-03-19 12:52:58,961] Trial 48 finished with value: 0.13350174316178703 and parameters: {'iterations': 1451, 'learning_rate': 0.06550136625740416, 'depth': 4, 'l2_leaf_reg': 1.5162069284107615, 'border_count': 104, 'random_strength': 0.001367673491504689, 'bagging_temperature': 0.1787606777470695}. Best is trial 30 with value: 0.13078553985762809.
[I 2025-03-19 12:52:59,883] Trial 49 finished with value: 0.12986119567304252 and parameters: {'iterations': 812, 'learning_rate': 0.05979948859085137, 'depth': 4, 'l2_leaf_reg': 3.1360289929700915, 'border_count': 173, 'random_strength': 0.0036705948488519546, 'bagging_temperature': 0.

0:	learn: 0.6612113	total: 5.61ms	remaining: 4.55s
1:	learn: 0.6238241	total: 12.3ms	remaining: 4.96s
2:	learn: 0.6014737	total: 17.5ms	remaining: 4.72s
3:	learn: 0.5781912	total: 23.1ms	remaining: 4.67s
4:	learn: 0.5631718	total: 30ms	remaining: 4.84s
5:	learn: 0.5450775	total: 35.6ms	remaining: 4.79s
6:	learn: 0.5325002	total: 42.8ms	remaining: 4.93s
7:	learn: 0.5215543	total: 49.4ms	remaining: 4.97s
8:	learn: 0.5114707	total: 57.8ms	remaining: 5.15s
9:	learn: 0.5030413	total: 65.4ms	remaining: 5.24s
10:	learn: 0.4960439	total: 72.8ms	remaining: 5.3s
11:	learn: 0.4896759	total: 80.7ms	remaining: 5.38s
12:	learn: 0.4809584	total: 92.9ms	remaining: 5.71s
13:	learn: 0.4750956	total: 102ms	remaining: 5.79s
14:	learn: 0.4684342	total: 108ms	remaining: 5.73s
15:	learn: 0.4632576	total: 114ms	remaining: 5.68s
16:	learn: 0.4585626	total: 120ms	remaining: 5.62s
17:	learn: 0.4546993	total: 127ms	remaining: 5.59s
18:	learn: 0.4512306	total: 131ms	remaining: 5.47s
19:	learn: 0.4470803	total: 136

176:	learn: 0.2677343	total: 1.08s	remaining: 3.87s
177:	learn: 0.2671867	total: 1.09s	remaining: 3.87s
178:	learn: 0.2664493	total: 1.09s	remaining: 3.86s
179:	learn: 0.2659259	total: 1.1s	remaining: 3.86s
180:	learn: 0.2653019	total: 1.1s	remaining: 3.85s
181:	learn: 0.2646318	total: 1.11s	remaining: 3.86s
182:	learn: 0.2637997	total: 1.12s	remaining: 3.85s
183:	learn: 0.2631826	total: 1.13s	remaining: 3.85s
184:	learn: 0.2628183	total: 1.14s	remaining: 3.85s
185:	learn: 0.2623956	total: 1.14s	remaining: 3.84s
186:	learn: 0.2618676	total: 1.15s	remaining: 3.85s
187:	learn: 0.2614207	total: 1.16s	remaining: 3.85s
188:	learn: 0.2607198	total: 1.17s	remaining: 3.86s
189:	learn: 0.2601288	total: 1.18s	remaining: 3.87s
190:	learn: 0.2595979	total: 1.19s	remaining: 3.87s
191:	learn: 0.2587420	total: 1.2s	remaining: 3.87s
192:	learn: 0.2580593	total: 1.2s	remaining: 3.86s
193:	learn: 0.2574069	total: 1.21s	remaining: 3.86s
194:	learn: 0.2568120	total: 1.22s	remaining: 3.85s
195:	learn: 0.25

...

1237:	learn: 0.0177714	total: 2m 58s	remaining: 1m 41s
1238:	learn: 0.0177430	total: 2m 58s	remaining: 1m 41s
1239:	learn: 0.0177187	total: 2m 58s	remaining: 1m 41s
1240:	learn: 0.0176958	total: 2m 58s	remaining: 1m 41s
1241:	learn: 0.0176735	total: 2m 58s	remaining: 1m 41s
1242:	learn: 0.0176361	total: 2m 59s	remaining: 1m 41s
1243:	learn: 0.0176122	total: 2m 59s	remaining: 1m 40s
1244:	learn: 0.0175892	total: 2m 59s	remaining: 1m 40s
1245:	learn: 0.0175661	total: 2m 59s	remaining: 1m 40s
1246:	learn: 0.0175414	total: 2m 59s	remaining: 1m 40s
1247:	learn: 0.0175132	total: 2m 59s	remaining: 1m 40s
1248:	learn: 0.0174961	total: 2m 59s	remaining: 1m 40s
1249:	learn: 0.0174676	total: 3m	remaining: 1m 40s
1250:	learn: 0.0174493	total: 3m	remaining: 1m 39s
1251:	learn: 0.0174260	total: 3m	remaining: 1m 39s
1252:	learn: 0.0173946	total: 3m	remaining: 1m 39s
1253:	learn: 0.0173641	total: 3m	remaining: 1m 39s
1254:	learn: 0.0173364	total: 3m	remaining: 1m 39s
1255:	learn: 0.0173053	total: 3m	r

1389:	learn: 0.0144293	total: 3m 20s	remaining: 1m 20s
1390:	learn: 0.0144200	total: 3m 20s	remaining: 1m 20s
1391:	learn: 0.0144010	total: 3m 21s	remaining: 1m 19s
1392:	learn: 0.0143886	total: 3m 21s	remaining: 1m 19s
1393:	learn: 0.0143689	total: 3m 21s	remaining: 1m 19s
1394:	learn: 0.0143493	total: 3m 21s	remaining: 1m 19s
1395:	learn: 0.0143373	total: 3m 21s	remaining: 1m 19s
1396:	learn: 0.0143183	total: 3m 21s	remaining: 1m 19s
1397:	learn: 0.0142992	total: 3m 21s	remaining: 1m 19s
1398:	learn: 0.0142797	total: 3m 22s	remaining: 1m 18s
1399:	learn: 0.0142564	total: 3m 22s	remaining: 1m 18s
1400:	learn: 0.0142336	total: 3m 22s	remaining: 1m 18s
1401:	learn: 0.0142186	total: 3m 22s	remaining: 1m 18s
1402:	learn: 0.0142031	total: 3m 22s	remaining: 1m 18s
1403:	learn: 0.0141894	total: 3m 22s	remaining: 1m 18s
1404:	learn: 0.0141712	total: 3m 22s	remaining: 1m 18s
1405:	learn: 0.0141565	total: 3m 23s	remaining: 1m 17s
1406:	learn: 0.0141398	total: 3m 23s	remaining: 1m 17s
1407:	lear

1541:	learn: 0.0120250	total: 3m 43s	remaining: 58.4s
1542:	learn: 0.0120146	total: 3m 43s	remaining: 58.2s
1543:	learn: 0.0120003	total: 3m 43s	remaining: 58.1s
1544:	learn: 0.0119821	total: 3m 43s	remaining: 57.9s
1545:	learn: 0.0119699	total: 3m 43s	remaining: 57.8s
1546:	learn: 0.0119608	total: 3m 44s	remaining: 57.6s
1547:	learn: 0.0119470	total: 3m 44s	remaining: 57.5s
1548:	learn: 0.0119333	total: 3m 44s	remaining: 57.3s
1549:	learn: 0.0119180	total: 3m 44s	remaining: 57.2s
1550:	learn: 0.0119060	total: 3m 44s	remaining: 57s
1551:	learn: 0.0118918	total: 3m 44s	remaining: 56.9s
1552:	learn: 0.0118793	total: 3m 44s	remaining: 56.7s
1553:	learn: 0.0118642	total: 3m 44s	remaining: 56.6s
1554:	learn: 0.0118537	total: 3m 45s	remaining: 56.4s
1555:	learn: 0.0118421	total: 3m 45s	remaining: 56.3s
1556:	learn: 0.0118260	total: 3m 45s	remaining: 56.1s
1557:	learn: 0.0118113	total: 3m 45s	remaining: 56s
1558:	learn: 0.0117991	total: 3m 45s	remaining: 55.9s
1559:	learn: 0.0117858	total: 3m

1695:	learn: 0.0101329	total: 4m 5s	remaining: 36.1s
1696:	learn: 0.0101239	total: 4m 5s	remaining: 35.9s
1697:	learn: 0.0101111	total: 4m 6s	remaining: 35.8s
1698:	learn: 0.0101006	total: 4m 6s	remaining: 35.6s
1699:	learn: 0.0100894	total: 4m 6s	remaining: 35.5s
1700:	learn: 0.0100781	total: 4m 6s	remaining: 35.4s
1701:	learn: 0.0100668	total: 4m 6s	remaining: 35.2s
1702:	learn: 0.0100547	total: 4m 6s	remaining: 35.1s
1703:	learn: 0.0100469	total: 4m 6s	remaining: 34.9s
1704:	learn: 0.0100385	total: 4m 7s	remaining: 34.8s
1705:	learn: 0.0100312	total: 4m 7s	remaining: 34.6s
1706:	learn: 0.0100211	total: 4m 7s	remaining: 34.5s
1707:	learn: 0.0100135	total: 4m 7s	remaining: 34.3s
1708:	learn: 0.0100022	total: 4m 7s	remaining: 34.2s
1709:	learn: 0.0099898	total: 4m 7s	remaining: 34s
1710:	learn: 0.0099796	total: 4m 7s	remaining: 33.9s
1711:	learn: 0.0099694	total: 4m 7s	remaining: 33.7s
1712:	learn: 0.0099594	total: 4m 8s	remaining: 33.6s
1713:	learn: 0.0099493	total: 4m 8s	remaining: 3

1849:	learn: 0.0087291	total: 4m 26s	remaining: 13.7s
1850:	learn: 0.0087210	total: 4m 26s	remaining: 13.5s
1851:	learn: 0.0087146	total: 4m 26s	remaining: 13.4s
1852:	learn: 0.0087050	total: 4m 26s	remaining: 13.2s
1853:	learn: 0.0086947	total: 4m 26s	remaining: 13.1s
1854:	learn: 0.0086855	total: 4m 27s	remaining: 13s
1855:	learn: 0.0086785	total: 4m 27s	remaining: 12.8s
1856:	learn: 0.0086728	total: 4m 27s	remaining: 12.7s
1857:	learn: 0.0086637	total: 4m 27s	remaining: 12.5s
1858:	learn: 0.0086562	total: 4m 27s	remaining: 12.4s
1859:	learn: 0.0086483	total: 4m 27s	remaining: 12.2s
1860:	learn: 0.0086423	total: 4m 27s	remaining: 12.1s
1861:	learn: 0.0086341	total: 4m 27s	remaining: 11.9s
1862:	learn: 0.0086236	total: 4m 28s	remaining: 11.8s
1863:	learn: 0.0086138	total: 4m 28s	remaining: 11.7s
1864:	learn: 0.0086057	total: 4m 28s	remaining: 11.5s
1865:	learn: 0.0085983	total: 4m 28s	remaining: 11.4s
1866:	learn: 0.0085912	total: 4m 28s	remaining: 11.2s
1867:	learn: 0.0085856	total: 

In [23]:
columns_to_include_women = [
 'Season',
 'DayNum',
 'T1_TeamID',
 'T1_Score',
 'T2_TeamID',
 'T2_Score',
 'location',
                      
#  'T1_Score_mean',
 'T1_FGM',
 'T1_FGA',
 'T1_FGM3',
 'T1_FGA3',
#  'T1_FTM',
#  'T1_FTA',
 'T1_OR',
#  'T1_DR',
 'T1_Ast',
 'T1_TO',
 'T1_Stl',
#  'T1_Blk',
 'T1_PF',
                      
#  'T1_opponent_Score',
 'T1_opponent_FGM',
 'T1_opponent_FGA',
 'T1_opponent_FGM3',
 'T1_opponent_FGA3',
#  'T1_opponent_FTM',
#  'T1_opponent_FTA',
 'T1_opponent_OR',
#  'T1_opponent_DR',
 'T1_opponent_Ast',
 'T1_opponent_TO',
 'T1_opponent_Stl',
#  'T1_opponent_Blk',
 'T1_opponent_PF',
 'T1_PointDiff',
                      
#  'T2_Score_mean',
 'T2_FGM',
 'T2_FGA',
 'T2_FGM3',
 'T2_FGA3',
#  'T2_FTM',
#  'T2_FTA',
 'T2_OR',
#  'T2_DR',
 'T2_Ast',
 'T2_TO',
 'T2_Stl',
#  'T2_Blk',
 'T2_PF',
                      
#  'T2_opponent_Score',
 'T2_opponent_FGM',
 'T2_opponent_FGA',
 'T2_opponent_FGM3',
 'T2_opponent_FGA3',
#  'T2_opponent_FTM',
#  'T2_opponent_FTA',
 'T2_opponent_OR',
#  'T2_opponent_DR',
 'T2_opponent_Ast',
 'T2_opponent_TO',
 'T2_opponent_Stl',
#  'T2_opponent_Blk',
 'T2_opponent_PF',
 'T2_PointDiff',
                      
 'T1_win_ratio_14d',
 'T2_win_ratio_14d',
 'T1_seed',
 'T2_seed',
 'Seed_diff',
 'TeamELO',
#  'CoachELO'
]
columns_to_include_men = [
 'Season',
 'DayNum',
 'T1_TeamID',
 'T1_Score',
 'T2_TeamID',
 'T2_Score',
 'location',
                      
#  'T1_Score_mean',
 'T1_FGM',
 'T1_FGA',
 'T1_FGM3',
 'T1_FGA3',
#  'T1_FTM',
#  'T1_FTA',
 'T1_OR',
#  'T1_DR',
 'T1_Ast',
 'T1_TO',
 'T1_Stl',
#  'T1_Blk',
 'T1_PF',
                      
#  'T1_opponent_Score',
 'T1_opponent_FGM',
 'T1_opponent_FGA',
 'T1_opponent_FGM3',
 'T1_opponent_FGA3',
#  'T1_opponent_FTM',
#  'T1_opponent_FTA',
 'T1_opponent_OR',
#  'T1_opponent_DR',
 'T1_opponent_Ast',
 'T1_opponent_TO',
 'T1_opponent_Stl',
#  'T1_opponent_Blk',
 'T1_opponent_PF',
 'T1_PointDiff',
                      
#  'T2_Score_mean',
 'T2_FGM',
 'T2_FGA',
 'T2_FGM3',
 'T2_FGA3',
#  'T2_FTM',
#  'T2_FTA',
 'T2_OR',
#  'T2_DR',
 'T2_Ast',
 'T2_TO',
 'T2_Stl',
#  'T2_Blk',
 'T2_PF',
                      
#  'T2_opponent_Score',
 'T2_opponent_FGM',
 'T2_opponent_FGA',
 'T2_opponent_FGM3',
 'T2_opponent_FGA3',
#  'T2_opponent_FTM',
#  'T2_opponent_FTA',
 'T2_opponent_OR',
#  'T2_opponent_DR',
 'T2_opponent_Ast',
 'T2_opponent_TO',
 'T2_opponent_Stl',
#  'T2_opponent_Blk',
 'T2_opponent_PF',
 'T2_PointDiff',
                      
 'T1_win_ratio_14d',
 'T2_win_ratio_14d',
 'T1_seed',
 'T2_seed',
 'Seed_diff',
 'TeamELO',
 'CoachELO'
]
print('All columns with ELO')
year_range = [i for i in range(2011, 2020)] + [i for i in range(2022, 2025)]
# ----------------------------------------------------------
# PARAMETERS TO CHANGE
start_season = 2003 # from which ponit should we begin creating data
season_years_list  = [[[i] for i in range(start_season, year+1)] for year in year_range] # at which seasons to look at when calculating team's stats
days_back = 150 # how many days back from the start of tourney to calculate team's stats per season
location_multiplier = [0.9, 1.1] # home penalty, away bonus ex.
win_ratio_days_back = 14 # how many days back from the tourney do we calculate win ratio
df_train_women_list, df_test_women_list, df_train_men_list, df_test_men_list = brier_for_all_years(year_range, season_years_list)
# ----------------------------------------------------------
x_train_women_list = []
x_test_women_list = []
x_train_men_list = []
x_test_men_list = []
y_train_women_list = []
y_test_women_list = []
y_train_men_list = []
y_test_men_list = []
brier_women_list = []
brier_men_list = []
brier_all_list = []
for i in range(len(year_range)):
    df_train_women_list[i] = df_train_women_list[i][columns_to_include_women]
    df_test_women_list[i] = df_test_women_list[i][columns_to_include_women]
    df_train_men_list[i] = df_train_men_list[i][columns_to_include_men]
    df_test_men_list[i] = df_test_men_list[i][columns_to_include_men]
    
    x_train_women, y_train_women = x_y_from_data_frame(df_train_women_list[i])
    x_test_women, y_test_women = x_y_from_data_frame(df_test_women_list[i])
    x_train_men, y_train_men = x_y_from_data_frame(df_train_men_list[i])
    x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list[i])

    x_train_women, y_train_women = clear_na_from_x_y(x_train_women.copy(), y_train_women.copy())
    x_test_women, y_test_women = clear_na_from_x_y(x_test_women.copy(), y_test_women.copy())
    x_train_men, y_train_men = clear_na_from_x_y(x_train_men.copy(), y_train_men.copy())
    x_test_men, y_test_men = clear_na_from_x_y(x_test_men.copy(), y_test_men.copy())
    
    x_train_women_list.append(x_train_women)
    x_test_women_list.append(x_test_women)
    x_train_men_list.append(x_train_men)
    x_test_men_list.append(x_test_men)
    
    y_train_women_list.append(y_train_women)
    y_test_women_list.append(y_test_women)
    y_train_men_list.append(y_train_men)
    y_test_men_list.append(y_test_men)
    
# -------------------------------------------------------

All columns with ELO


In [43]:
j = 6
print('Fitting catboost on years: ', year_range[0], '-', year_range[j])
x_train_women = x_train_women_list[j]
y_train_women = y_train_women_list[j]
x_test_women = x_test_women_list[j]
y_test_women = y_test_women_list[j]
best_catboost_women = CatBoostClassifier(**best_params_women)
best_catboost_women.fit(x_train_women, y_train_women)

Fitting catboost on years:  2011 - 2017
0:	learn: 0.6555726	total: 3.15ms	remaining: 2.56s
1:	learn: 0.6255551	total: 7.51ms	remaining: 3.04s
2:	learn: 0.5978938	total: 11.4ms	remaining: 3.06s
3:	learn: 0.5718082	total: 15.6ms	remaining: 3.14s
4:	learn: 0.5505081	total: 18.5ms	remaining: 2.99s
5:	learn: 0.5330118	total: 22.6ms	remaining: 3.03s
6:	learn: 0.5177169	total: 25.7ms	remaining: 2.95s
7:	learn: 0.5063828	total: 28.5ms	remaining: 2.86s
8:	learn: 0.4948366	total: 31.4ms	remaining: 2.8s
9:	learn: 0.4844825	total: 34.3ms	remaining: 2.75s
10:	learn: 0.4747490	total: 37.7ms	remaining: 2.75s
11:	learn: 0.4669719	total: 40.8ms	remaining: 2.72s
12:	learn: 0.4593776	total: 43.6ms	remaining: 2.68s
13:	learn: 0.4520785	total: 46.6ms	remaining: 2.65s
14:	learn: 0.4426322	total: 49.5ms	remaining: 2.63s
15:	learn: 0.4353699	total: 52.6ms	remaining: 2.61s
16:	learn: 0.4298285	total: 55.5ms	remaining: 2.6s
17:	learn: 0.4220348	total: 58.4ms	remaining: 2.58s
18:	learn: 0.4175567	total: 61.3ms	r

165:	learn: 0.2108696	total: 515ms	remaining: 2s
166:	learn: 0.2103739	total: 518ms	remaining: 2s
167:	learn: 0.2093316	total: 520ms	remaining: 1.99s
168:	learn: 0.2080394	total: 523ms	remaining: 1.99s
169:	learn: 0.2074240	total: 526ms	remaining: 1.99s
170:	learn: 0.2063385	total: 529ms	remaining: 1.98s
171:	learn: 0.2056493	total: 532ms	remaining: 1.98s
172:	learn: 0.2044092	total: 535ms	remaining: 1.98s
173:	learn: 0.2042696	total: 539ms	remaining: 1.98s
174:	learn: 0.2033453	total: 543ms	remaining: 1.98s
175:	learn: 0.2023350	total: 546ms	remaining: 1.97s
176:	learn: 0.2012856	total: 550ms	remaining: 1.97s
177:	learn: 0.2006555	total: 552ms	remaining: 1.97s
178:	learn: 0.1996255	total: 555ms	remaining: 1.96s
179:	learn: 0.1985344	total: 559ms	remaining: 1.96s
180:	learn: 0.1971562	total: 562ms	remaining: 1.96s
181:	learn: 0.1962941	total: 565ms	remaining: 1.96s
182:	learn: 0.1959751	total: 568ms	remaining: 1.95s
183:	learn: 0.1951731	total: 571ms	remaining: 1.95s
184:	learn: 0.1950

353:	learn: 0.1028201	total: 1.07s	remaining: 1.38s
354:	learn: 0.1027673	total: 1.07s	remaining: 1.38s
355:	learn: 0.1023677	total: 1.07s	remaining: 1.37s
356:	learn: 0.1018423	total: 1.08s	remaining: 1.37s
357:	learn: 0.1017590	total: 1.08s	remaining: 1.37s
358:	learn: 0.1012231	total: 1.08s	remaining: 1.36s
359:	learn: 0.1008624	total: 1.08s	remaining: 1.36s
360:	learn: 0.1005392	total: 1.09s	remaining: 1.36s
361:	learn: 0.1003329	total: 1.09s	remaining: 1.36s
362:	learn: 0.0998075	total: 1.09s	remaining: 1.35s
363:	learn: 0.0992671	total: 1.1s	remaining: 1.35s
364:	learn: 0.0987248	total: 1.1s	remaining: 1.35s
365:	learn: 0.0982211	total: 1.1s	remaining: 1.35s
366:	learn: 0.0976764	total: 1.11s	remaining: 1.34s
367:	learn: 0.0975885	total: 1.11s	remaining: 1.34s
368:	learn: 0.0973620	total: 1.11s	remaining: 1.34s
369:	learn: 0.0968657	total: 1.12s	remaining: 1.33s
370:	learn: 0.0968352	total: 1.12s	remaining: 1.33s
371:	learn: 0.0964225	total: 1.12s	remaining: 1.33s
372:	learn: 0.0

544:	learn: 0.0571124	total: 1.62s	remaining: 795ms
545:	learn: 0.0569923	total: 1.63s	remaining: 792ms
546:	learn: 0.0567773	total: 1.63s	remaining: 789ms
547:	learn: 0.0567314	total: 1.63s	remaining: 786ms
548:	learn: 0.0566168	total: 1.63s	remaining: 783ms
549:	learn: 0.0565576	total: 1.64s	remaining: 780ms
550:	learn: 0.0564178	total: 1.64s	remaining: 777ms
551:	learn: 0.0562144	total: 1.64s	remaining: 774ms
552:	learn: 0.0562013	total: 1.65s	remaining: 771ms
553:	learn: 0.0561621	total: 1.65s	remaining: 769ms
554:	learn: 0.0560810	total: 1.65s	remaining: 766ms
555:	learn: 0.0560723	total: 1.66s	remaining: 763ms
556:	learn: 0.0559218	total: 1.66s	remaining: 760ms
557:	learn: 0.0557700	total: 1.66s	remaining: 757ms
558:	learn: 0.0556472	total: 1.67s	remaining: 754ms
559:	learn: 0.0553727	total: 1.67s	remaining: 751ms
560:	learn: 0.0553337	total: 1.67s	remaining: 748ms
561:	learn: 0.0551923	total: 1.67s	remaining: 745ms
562:	learn: 0.0549393	total: 1.68s	remaining: 742ms
563:	learn: 

740:	learn: 0.0349720	total: 2.35s	remaining: 225ms
741:	learn: 0.0349505	total: 2.35s	remaining: 222ms
742:	learn: 0.0348303	total: 2.36s	remaining: 219ms
743:	learn: 0.0347571	total: 2.36s	remaining: 216ms
744:	learn: 0.0346558	total: 2.37s	remaining: 213ms
745:	learn: 0.0345981	total: 2.37s	remaining: 210ms
746:	learn: 0.0345844	total: 2.37s	remaining: 206ms
747:	learn: 0.0345487	total: 2.38s	remaining: 203ms
748:	learn: 0.0345012	total: 2.38s	remaining: 200ms
749:	learn: 0.0344952	total: 2.39s	remaining: 197ms
750:	learn: 0.0344445	total: 2.39s	remaining: 194ms
751:	learn: 0.0343934	total: 2.39s	remaining: 191ms
752:	learn: 0.0342052	total: 2.4s	remaining: 188ms
753:	learn: 0.0341293	total: 2.4s	remaining: 185ms
754:	learn: 0.0341201	total: 2.41s	remaining: 182ms
755:	learn: 0.0339975	total: 2.41s	remaining: 179ms
756:	learn: 0.0339150	total: 2.41s	remaining: 175ms
757:	learn: 0.0338748	total: 2.42s	remaining: 172ms
758:	learn: 0.0337607	total: 2.42s	remaining: 169ms
759:	learn: 0.

In [51]:
print('Fitting catboost on years: ', year_range[-1], '-', year_range[j])
x_train_men = x_train_men_list[j]
y_train_men = y_train_men_list[j]
x_test_men = x_test_men_list[j]
y_test_men = y_test_men_list[j]
best_catboost_men = CatBoostClassifier(**best_params_men)
best_catboost_men.fit(x_train_men, y_train_men)

Fitting catboost on years:  2024 - 2017
0:	learn: 0.6832118	total: 113ms	remaining: 3m 40s
1:	learn: 0.6755180	total: 227ms	remaining: 3m 40s
2:	learn: 0.6672451	total: 337ms	remaining: 3m 38s
3:	learn: 0.6579877	total: 446ms	remaining: 3m 36s
4:	learn: 0.6497830	total: 560ms	remaining: 3m 37s
5:	learn: 0.6418970	total: 683ms	remaining: 3m 40s
6:	learn: 0.6348839	total: 832ms	remaining: 3m 50s
7:	learn: 0.6278196	total: 957ms	remaining: 3m 51s
8:	learn: 0.6207046	total: 1.09s	remaining: 3m 54s
9:	learn: 0.6139511	total: 1.22s	remaining: 3m 56s
10:	learn: 0.6068221	total: 1.37s	remaining: 4m
11:	learn: 0.6004771	total: 1.51s	remaining: 4m 3s
12:	learn: 0.5934763	total: 1.63s	remaining: 4m 1s
13:	learn: 0.5852149	total: 1.75s	remaining: 4m 1s
14:	learn: 0.5781513	total: 1.89s	remaining: 4m 3s
15:	learn: 0.5714083	total: 2.06s	remaining: 4m 7s
16:	learn: 0.5650296	total: 2.19s	remaining: 4m 8s
17:	learn: 0.5592957	total: 2.36s	remaining: 4m 12s
18:	learn: 0.5537156	total: 2.5s	remaining: 

159:	learn: 0.2026715	total: 21.9s	remaining: 4m 4s
160:	learn: 0.2019350	total: 22s	remaining: 4m 3s
161:	learn: 0.2010638	total: 22.1s	remaining: 4m 3s
162:	learn: 0.1997098	total: 22.3s	remaining: 4m 3s
163:	learn: 0.1986448	total: 22.4s	remaining: 4m 3s
164:	learn: 0.1975632	total: 22.5s	remaining: 4m 2s
165:	learn: 0.1962529	total: 22.6s	remaining: 4m 2s
166:	learn: 0.1952672	total: 22.8s	remaining: 4m 2s
167:	learn: 0.1943245	total: 22.9s	remaining: 4m 2s
168:	learn: 0.1935689	total: 23s	remaining: 4m 1s
169:	learn: 0.1927911	total: 23.1s	remaining: 4m 1s
170:	learn: 0.1920442	total: 23.3s	remaining: 4m 1s
171:	learn: 0.1915140	total: 23.4s	remaining: 4m 1s
172:	learn: 0.1908766	total: 23.5s	remaining: 4m
173:	learn: 0.1896721	total: 23.6s	remaining: 4m
174:	learn: 0.1887222	total: 23.8s	remaining: 4m
175:	learn: 0.1879433	total: 23.9s	remaining: 4m
176:	learn: 0.1869243	total: 24s	remaining: 3m 59s
177:	learn: 0.1864060	total: 24.1s	remaining: 3m 59s
178:	learn: 0.1854390	total:

315:	learn: 0.1003712	total: 42.8s	remaining: 3m 40s
316:	learn: 0.0999980	total: 42.9s	remaining: 3m 40s
317:	learn: 0.0995033	total: 43.1s	remaining: 3m 40s
318:	learn: 0.0990189	total: 43.2s	remaining: 3m 40s
319:	learn: 0.0985089	total: 43.3s	remaining: 3m 40s
320:	learn: 0.0980636	total: 43.5s	remaining: 3m 39s
321:	learn: 0.0975567	total: 43.6s	remaining: 3m 39s
322:	learn: 0.0970075	total: 43.7s	remaining: 3m 39s
323:	learn: 0.0966946	total: 43.9s	remaining: 3m 39s
324:	learn: 0.0961668	total: 44s	remaining: 3m 39s
325:	learn: 0.0956627	total: 44.1s	remaining: 3m 39s
326:	learn: 0.0950463	total: 44.2s	remaining: 3m 38s
327:	learn: 0.0946917	total: 44.4s	remaining: 3m 38s
328:	learn: 0.0943405	total: 44.5s	remaining: 3m 38s
329:	learn: 0.0938602	total: 44.6s	remaining: 3m 38s
330:	learn: 0.0933612	total: 44.7s	remaining: 3m 38s
331:	learn: 0.0930041	total: 44.9s	remaining: 3m 37s
332:	learn: 0.0926820	total: 45s	remaining: 3m 37s
333:	learn: 0.0924557	total: 45.1s	remaining: 3m 3

471:	learn: 0.0535747	total: 1m 3s	remaining: 3m 19s
472:	learn: 0.0534140	total: 1m 4s	remaining: 3m 19s
473:	learn: 0.0532734	total: 1m 4s	remaining: 3m 19s
474:	learn: 0.0531142	total: 1m 4s	remaining: 3m 19s
475:	learn: 0.0528944	total: 1m 4s	remaining: 3m 19s
476:	learn: 0.0526289	total: 1m 4s	remaining: 3m 19s
477:	learn: 0.0524659	total: 1m 4s	remaining: 3m 18s
478:	learn: 0.0522276	total: 1m 4s	remaining: 3m 18s
479:	learn: 0.0520475	total: 1m 5s	remaining: 3m 18s
480:	learn: 0.0518769	total: 1m 5s	remaining: 3m 18s
481:	learn: 0.0516349	total: 1m 5s	remaining: 3m 18s
482:	learn: 0.0513842	total: 1m 5s	remaining: 3m 18s
483:	learn: 0.0512447	total: 1m 5s	remaining: 3m 18s
484:	learn: 0.0510237	total: 1m 5s	remaining: 3m 17s
485:	learn: 0.0508529	total: 1m 5s	remaining: 3m 17s
486:	learn: 0.0506212	total: 1m 6s	remaining: 3m 17s
487:	learn: 0.0504402	total: 1m 6s	remaining: 3m 17s
488:	learn: 0.0502392	total: 1m 6s	remaining: 3m 17s
489:	learn: 0.0501436	total: 1m 6s	remaining: 

625:	learn: 0.0327383	total: 1m 25s	remaining: 3m
626:	learn: 0.0326404	total: 1m 26s	remaining: 3m
627:	learn: 0.0325423	total: 1m 26s	remaining: 3m
628:	learn: 0.0324283	total: 1m 26s	remaining: 3m
629:	learn: 0.0323499	total: 1m 26s	remaining: 3m
630:	learn: 0.0322852	total: 1m 26s	remaining: 3m
631:	learn: 0.0322003	total: 1m 26s	remaining: 3m
632:	learn: 0.0321072	total: 1m 26s	remaining: 2m 59s
633:	learn: 0.0320202	total: 1m 26s	remaining: 2m 59s
634:	learn: 0.0319007	total: 1m 27s	remaining: 2m 59s
635:	learn: 0.0317958	total: 1m 27s	remaining: 2m 59s
636:	learn: 0.0317123	total: 1m 27s	remaining: 2m 59s
637:	learn: 0.0316189	total: 1m 27s	remaining: 2m 59s
638:	learn: 0.0315277	total: 1m 27s	remaining: 2m 59s
639:	learn: 0.0314230	total: 1m 27s	remaining: 2m 59s
640:	learn: 0.0313585	total: 1m 28s	remaining: 2m 59s
641:	learn: 0.0312584	total: 1m 28s	remaining: 2m 59s
642:	learn: 0.0311934	total: 1m 28s	remaining: 2m 59s
643:	learn: 0.0311195	total: 1m 28s	remaining: 2m 58s
64

...

1236:	learn: 0.0103458	total: 2m 44s	remaining: 1m 34s
1237:	learn: 0.0103285	total: 2m 44s	remaining: 1m 34s
1238:	learn: 0.0103126	total: 2m 44s	remaining: 1m 34s
1239:	learn: 0.0102967	total: 2m 45s	remaining: 1m 33s
1240:	learn: 0.0102827	total: 2m 45s	remaining: 1m 33s
1241:	learn: 0.0102680	total: 2m 45s	remaining: 1m 33s
1242:	learn: 0.0102522	total: 2m 45s	remaining: 1m 33s
1243:	learn: 0.0102359	total: 2m 45s	remaining: 1m 33s
1244:	learn: 0.0102218	total: 2m 45s	remaining: 1m 33s
1245:	learn: 0.0102089	total: 2m 46s	remaining: 1m 33s
1246:	learn: 0.0101944	total: 2m 46s	remaining: 1m 33s
1247:	learn: 0.0101799	total: 2m 46s	remaining: 1m 32s
1248:	learn: 0.0101677	total: 2m 46s	remaining: 1m 32s
1249:	learn: 0.0101542	total: 2m 46s	remaining: 1m 32s
1250:	learn: 0.0101421	total: 2m 46s	remaining: 1m 32s
1251:	learn: 0.0101289	total: 2m 46s	remaining: 1m 32s
1252:	learn: 0.0101122	total: 2m 47s	remaining: 1m 32s
1253:	learn: 0.0100980	total: 2m 47s	remaining: 1m 32s
1254:	lear

1388:	learn: 0.0085380	total: 3m 5s	remaining: 1m 14s
1389:	learn: 0.0085306	total: 3m 5s	remaining: 1m 14s
1390:	learn: 0.0085223	total: 3m 5s	remaining: 1m 14s
1391:	learn: 0.0085129	total: 3m 6s	remaining: 1m 13s
1392:	learn: 0.0085039	total: 3m 6s	remaining: 1m 13s
1393:	learn: 0.0084949	total: 3m 6s	remaining: 1m 13s
1394:	learn: 0.0084827	total: 3m 6s	remaining: 1m 13s
1395:	learn: 0.0084728	total: 3m 6s	remaining: 1m 13s
1396:	learn: 0.0084642	total: 3m 6s	remaining: 1m 13s
1397:	learn: 0.0084525	total: 3m 6s	remaining: 1m 13s
1398:	learn: 0.0084407	total: 3m 7s	remaining: 1m 13s
1399:	learn: 0.0084290	total: 3m 7s	remaining: 1m 12s
1400:	learn: 0.0084181	total: 3m 7s	remaining: 1m 12s
1401:	learn: 0.0084101	total: 3m 7s	remaining: 1m 12s
1402:	learn: 0.0084017	total: 3m 7s	remaining: 1m 12s
1403:	learn: 0.0083910	total: 3m 7s	remaining: 1m 12s
1404:	learn: 0.0083810	total: 3m 7s	remaining: 1m 12s
1405:	learn: 0.0083731	total: 3m 7s	remaining: 1m 12s
1406:	learn: 0.0083630	total

1542:	learn: 0.0071993	total: 3m 26s	remaining: 53.8s
1543:	learn: 0.0071917	total: 3m 26s	remaining: 53.7s
1544:	learn: 0.0071844	total: 3m 26s	remaining: 53.6s
1545:	learn: 0.0071762	total: 3m 26s	remaining: 53.4s
1546:	learn: 0.0071698	total: 3m 27s	remaining: 53.3s
1547:	learn: 0.0071640	total: 3m 27s	remaining: 53.1s
1548:	learn: 0.0071552	total: 3m 27s	remaining: 53s
1549:	learn: 0.0071493	total: 3m 27s	remaining: 52.9s
1550:	learn: 0.0071417	total: 3m 27s	remaining: 52.8s
1551:	learn: 0.0071342	total: 3m 27s	remaining: 52.6s
1552:	learn: 0.0071256	total: 3m 27s	remaining: 52.5s
1553:	learn: 0.0071193	total: 3m 28s	remaining: 52.4s
1554:	learn: 0.0071110	total: 3m 28s	remaining: 52.2s
1555:	learn: 0.0071022	total: 3m 28s	remaining: 52.1s
1556:	learn: 0.0070938	total: 3m 28s	remaining: 52s
1557:	learn: 0.0070867	total: 3m 28s	remaining: 51.8s
1558:	learn: 0.0070806	total: 3m 28s	remaining: 51.7s
1559:	learn: 0.0070734	total: 3m 28s	remaining: 51.5s
1560:	learn: 0.0070663	total: 3m

1696:	learn: 0.0061946	total: 3m 46s	remaining: 33.1s
1697:	learn: 0.0061892	total: 3m 46s	remaining: 32.9s
1698:	learn: 0.0061830	total: 3m 46s	remaining: 32.8s
1699:	learn: 0.0061773	total: 3m 46s	remaining: 32.7s
1700:	learn: 0.0061719	total: 3m 46s	remaining: 32.5s
1701:	learn: 0.0061652	total: 3m 46s	remaining: 32.4s
1702:	learn: 0.0061595	total: 3m 47s	remaining: 32.3s
1703:	learn: 0.0061529	total: 3m 47s	remaining: 32.1s
1704:	learn: 0.0061462	total: 3m 47s	remaining: 32s
1705:	learn: 0.0061408	total: 3m 47s	remaining: 31.9s
1706:	learn: 0.0061336	total: 3m 47s	remaining: 31.7s
1707:	learn: 0.0061283	total: 3m 47s	remaining: 31.6s
1708:	learn: 0.0061230	total: 3m 47s	remaining: 31.5s
1709:	learn: 0.0061157	total: 3m 47s	remaining: 31.3s
1710:	learn: 0.0061087	total: 3m 48s	remaining: 31.2s
1711:	learn: 0.0061022	total: 3m 48s	remaining: 31.1s
1712:	learn: 0.0060980	total: 3m 48s	remaining: 30.9s
1713:	learn: 0.0060924	total: 3m 48s	remaining: 30.8s
1714:	learn: 0.0060876	total: 

1851:	learn: 0.0053855	total: 4m 10s	remaining: 12.6s
1852:	learn: 0.0053802	total: 4m 10s	remaining: 12.4s
1853:	learn: 0.0053759	total: 4m 10s	remaining: 12.3s
1854:	learn: 0.0053713	total: 4m 10s	remaining: 12.2s
1855:	learn: 0.0053680	total: 4m 10s	remaining: 12s
1856:	learn: 0.0053625	total: 4m 10s	remaining: 11.9s
1857:	learn: 0.0053578	total: 4m 10s	remaining: 11.8s
1858:	learn: 0.0053528	total: 4m 11s	remaining: 11.6s
1859:	learn: 0.0053488	total: 4m 11s	remaining: 11.5s
1860:	learn: 0.0053437	total: 4m 11s	remaining: 11.4s
1861:	learn: 0.0053399	total: 4m 11s	remaining: 11.2s
1862:	learn: 0.0053345	total: 4m 12s	remaining: 11.1s
1863:	learn: 0.0053322	total: 4m 12s	remaining: 11s
1864:	learn: 0.0053279	total: 4m 12s	remaining: 10.8s
1865:	learn: 0.0053239	total: 4m 12s	remaining: 10.7s
1866:	learn: 0.0053204	total: 4m 12s	remaining: 10.6s
1867:	learn: 0.0053155	total: 4m 12s	remaining: 10.4s
1868:	learn: 0.0053111	total: 4m 13s	remaining: 10.3s
1869:	learn: 0.0053075	total: 4m

In [53]:
brier_women_list = []
brier_men_list = []
brier_all_list = []
for i in range(6+1, len(year_range)):
    print("Predictions for season ", year_range[i])
    x_train_women = x_train_women_list[i]
    y_train_women = y_train_women_list[i]
    x_test_women = x_test_women_list[i]
    y_test_women = y_test_women_list[i]
    
    y_prob_women = best_catboost_women.predict_proba(x_test_women)[:, 1]
    score = brier_score_loss(y_test_women, y_prob_women)
    brier_women_list.append(score)
    print('LogisticRegressionCV for women trained on women', score)
    # ----------------------------------------------
    x_train_men = x_train_men_list[i]
    y_train_men = y_train_men_list[i]
    x_test_men = x_test_men_list[i]
    y_test_men = y_test_men_list[i]
    
    y_prob_men = best_catboost_men.predict_proba(x_test_men)[:, 1]
    score = brier_score_loss(y_test_men, y_prob_men)
    brier_men_list.append(score)
    print('LogisticRegressionCV for men trained on men', score)
    # ----------------------------------------------
    y_prob = np.concatenate((y_prob_women, y_prob_men))
    y_test = np.concatenate((y_test_women, y_test_men))
    score = brier_score_loss(y_test, y_prob)
    brier_all_list.append(score)
    print('LogisticRegressionCV for all trained on women and men separately', score, '<---')
    print()
    # ----------------------------------------------
print()
print('The mean score when trained on women and tested on women was: ', np.mean(brier_women_list), 
      'with a std of ', np.std(brier_women_list))
print('The mean score when trained on men and tested on men was: ', np.mean(brier_men_list), 
      'with a std of ', np.std(brier_men_list))
print('The mean score when trained separately and tested on both was: ', np.mean(brier_all_list), 
      'with a std of ', np.std(brier_all_list))

Predictions for season  2018
LogisticRegressionCV for women trained on women 0.1840709540005396
LogisticRegressionCV for men trained on men 0.23372381341821788
LogisticRegressionCV for all trained on women and men separately 0.20966127385426614 <---

Predictions for season  2019
LogisticRegressionCV for women trained on women 0.20186610718568718
LogisticRegressionCV for men trained on men 0.1721897605009993
LogisticRegressionCV for all trained on women and men separately 0.18657137466357882 <---

Predictions for season  2022
LogisticRegressionCV for women trained on women 0.19276179468366708
LogisticRegressionCV for men trained on men 0.26235247928697686
LogisticRegressionCV for all trained on women and men separately 0.22781488026163058 <---

Predictions for season  2023
LogisticRegressionCV for women trained on women 0.18283729439300728
LogisticRegressionCV for men trained on men 0.23052883994243364
LogisticRegressionCV for all trained on women and men separately 0.20737424898727733 

In [20]:
best_catboost_men(**best_params)

TypeError: 'CatBoostClassifier' object is not callable

## How trained models perform

In [ ]:
# Results for model fitted on seasons 2011-2022

# Predictions for season  2011
# LogisticRegressionCV for women trained on women 8.6234908368925e-05
# LogisticRegressionCV for men trained on men 1.195440151810999e-05
# LogisticRegressionCV for all trained on women and men separately 4.795187791504342e-05 <---

# Predictions for season  2012
# LogisticRegressionCV for women trained on women 9.603887978799821e-05
# LogisticRegressionCV for men trained on men 1.0096747241159672e-05
# LogisticRegressionCV for all trained on women and men separately 5.1114583229423516e-05 <---

# Predictions for season  2013
# LogisticRegressionCV for women trained on women 0.00011929294872597823
# LogisticRegressionCV for men trained on men 1.1039176707217715e-05
# LogisticRegressionCV for all trained on women and men separately 6.350062007015553e-05 <---

# Predictions for season  2014
# LogisticRegressionCV for women trained on women 9.400880734527388e-05
# LogisticRegressionCV for men trained on men 7.470347068399079e-06
# LogisticRegressionCV for all trained on women and men separately 4.940821627949996e-05 <---

# Predictions for season  2015
# LogisticRegressionCV for women trained on women 0.00011703312235407021
# LogisticRegressionCV for men trained on men 6.654722660794484e-06
# LogisticRegressionCV for all trained on women and men separately 6.014579328138196e-05 <---

# Predictions for season  2016
# LogisticRegressionCV for women trained on women 0.00014535900721294036
# LogisticRegressionCV for men trained on men 7.096983417623299e-06
# LogisticRegressionCV for all trained on women and men separately 7.258952100487874e-05 <---

# Predictions for season  2017
# LogisticRegressionCV for women trained on women 9.871652446079126e-05
# LogisticRegressionCV for men trained on men 8.664143297518513e-06
# LogisticRegressionCV for all trained on women and men separately 5.230491263048917e-05 <---

# Predictions for season  2018
# LogisticRegressionCV for women trained on women 0.00011748549950079368
# LogisticRegressionCV for men trained on men 1.0139029733009277e-05
# LogisticRegressionCV for all trained on women and men separately 6.216078046662786e-05 <---

# Predictions for season  2019
# LogisticRegressionCV for women trained on women 9.054570663778677e-05
# LogisticRegressionCV for men trained on men 8.404029187726703e-06
# LogisticRegressionCV for all trained on women and men separately 4.821114979814043e-05 <---

# Predictions for season  2022
# LogisticRegressionCV for women trained on women 0.0001232312094622996
# LogisticRegressionCV for men trained on men 5.935757624128796e-06
# LogisticRegressionCV for all trained on women and men separately 6.414905594381357e-05 <---

# Predictions for season  2023
# LogisticRegressionCV for women trained on women 0.18229452535339608
# LogisticRegressionCV for men trained on men 0.23188417352571136
# LogisticRegressionCV for all trained on women and men separately 0.20780803999277572 <---

# Predictions for season  2024
# LogisticRegressionCV for women trained on women 0.12897573922737807
# LogisticRegressionCV for men trained on men 0.22752791162728128
# LogisticRegressionCV for all trained on women and men separately 0.17825182542732967 <---


# The mean score when trained on women and tested on women was:  0.026029850932885918 with a std of  0.0589742310212988
# The mean score when trained on men and tested on men was:  0.038291628374287366 with a std of  0.08560774675782466
# The mean score when trained separately and tested on both was:  0.03221928349422707 with a std of  0.0721693271172961

## Training and testing logistic regression

In [54]:
columns_to_include_women = [
 'Season',
 'DayNum',
 'T1_TeamID',
 'T1_Score',
 'T2_TeamID',
 'T2_Score',
 'location',
                      
#  'T1_Score_mean',
 'T1_FGM',
 'T1_FGA',
 'T1_FGM3',
 'T1_FGA3',
#  'T1_FTM',
#  'T1_FTA',
 'T1_OR',
#  'T1_DR',
 'T1_Ast',
 'T1_TO',
 'T1_Stl',
#  'T1_Blk',
 'T1_PF',
                      
#  'T1_opponent_Score',
 'T1_opponent_FGM',
 'T1_opponent_FGA',
 'T1_opponent_FGM3',
 'T1_opponent_FGA3',
#  'T1_opponent_FTM',
#  'T1_opponent_FTA',
 'T1_opponent_OR',
#  'T1_opponent_DR',
 'T1_opponent_Ast',
 'T1_opponent_TO',
 'T1_opponent_Stl',
#  'T1_opponent_Blk',
 'T1_opponent_PF',
 'T1_PointDiff',
                      
#  'T2_Score_mean',
 'T2_FGM',
 'T2_FGA',
 'T2_FGM3',
 'T2_FGA3',
#  'T2_FTM',
#  'T2_FTA',
 'T2_OR',
#  'T2_DR',
 'T2_Ast',
 'T2_TO',
 'T2_Stl',
#  'T2_Blk',
 'T2_PF',
                      
#  'T2_opponent_Score',
 'T2_opponent_FGM',
 'T2_opponent_FGA',
 'T2_opponent_FGM3',
 'T2_opponent_FGA3',
#  'T2_opponent_FTM',
#  'T2_opponent_FTA',
 'T2_opponent_OR',
#  'T2_opponent_DR',
 'T2_opponent_Ast',
 'T2_opponent_TO',
 'T2_opponent_Stl',
#  'T2_opponent_Blk',
 'T2_opponent_PF',
 'T2_PointDiff',
                      
 'T1_win_ratio_14d',
 'T2_win_ratio_14d',
 'T1_seed',
 'T2_seed',
 'Seed_diff',
 'TeamELO',
#  'CoachELO'
]
columns_to_include_men = [
 'Season',
 'DayNum',
 'T1_TeamID',
 'T1_Score',
 'T2_TeamID',
 'T2_Score',
 'location',
                      
#  'T1_Score_mean',
 'T1_FGM',
 'T1_FGA',
 'T1_FGM3',
 'T1_FGA3',
#  'T1_FTM',
#  'T1_FTA',
 'T1_OR',
#  'T1_DR',
 'T1_Ast',
 'T1_TO',
 'T1_Stl',
#  'T1_Blk',
 'T1_PF',
                      
#  'T1_opponent_Score',
 'T1_opponent_FGM',
 'T1_opponent_FGA',
 'T1_opponent_FGM3',
 'T1_opponent_FGA3',
#  'T1_opponent_FTM',
#  'T1_opponent_FTA',
 'T1_opponent_OR',
#  'T1_opponent_DR',
 'T1_opponent_Ast',
 'T1_opponent_TO',
 'T1_opponent_Stl',
#  'T1_opponent_Blk',
 'T1_opponent_PF',
 'T1_PointDiff',
                      
#  'T2_Score_mean',
 'T2_FGM',
 'T2_FGA',
 'T2_FGM3',
 'T2_FGA3',
#  'T2_FTM',
#  'T2_FTA',
 'T2_OR',
#  'T2_DR',
 'T2_Ast',
 'T2_TO',
 'T2_Stl',
#  'T2_Blk',
 'T2_PF',
                      
#  'T2_opponent_Score',
 'T2_opponent_FGM',
 'T2_opponent_FGA',
 'T2_opponent_FGM3',
 'T2_opponent_FGA3',
#  'T2_opponent_FTM',
#  'T2_opponent_FTA',
 'T2_opponent_OR',
#  'T2_opponent_DR',
 'T2_opponent_Ast',
 'T2_opponent_TO',
 'T2_opponent_Stl',
#  'T2_opponent_Blk',
 'T2_opponent_PF',
 'T2_PointDiff',
                      
 'T1_win_ratio_14d',
 'T2_win_ratio_14d',
 'T1_seed',
 'T2_seed',
 'Seed_diff',
 'TeamELO',
 'CoachELO'
]

year_range = [2023]
start_season = 2003 # from which ponit should we begin creating data
season_years_list  = [[[i] for i in range(start_season, year+1)] for year in year_range] # at which seasons to look at when calculating team's stats
days_back = 150 # how many days back from the start of tourney to calculate team's stats per season
location_multiplier = [0.9, 1.1] # home penalty, away bonus ex.
win_ratio_days_back = 14 # how many days back from the tourney do we calculate win ratio
maximum_favoured_seed = 0
# -------------------------------------------
df_train_women_list, df_test_women_list, df_train_men_list, df_test_men_list = brier_for_all_years(year_range, season_years_list)
x_train_women_list = []
x_test_women_list = []
x_train_men_list = []
x_test_men_list = []
y_train_women_list = []
y_test_women_list = []
y_train_men_list = []
y_test_men_list = []
for i in range(len(year_range)):

    if len(year_range) > 1:
        df_train_women_list[i] = df_train_women_list[i][columns_to_include_women]
        df_test_women_list[i] = df_test_women_list[i][columns_to_include_women]
        df_train_men_list[i] = df_train_men_list[i][columns_to_include_men]
        df_test_men_list[i] = df_test_men_list[i][columns_to_include_men]
        
        x_train_women, y_train_women = x_y_from_data_frame(df_train_women_list[i])
        x_test_women, y_test_women = x_y_from_data_frame(df_test_women_list[i])
        x_train_men, y_train_men = x_y_from_data_frame(df_train_men_list[i])
        x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list[i])

        x_train_women, y_train_women = clear_na_from_x_y(x_train_women.copy(), y_train_women.copy())
        x_test_women, y_test_women = clear_na_from_x_y(x_test_women.copy(), y_test_women.copy())
        x_train_men, y_train_men = clear_na_from_x_y(x_train_men.copy(), y_train_men.copy())
        x_test_men, y_test_men = clear_na_from_x_y(x_test_men.copy(), y_test_men.copy())
        
        x_train_women_list.append(x_train_women)
        x_test_women_list.append(x_test_women)
        x_train_men_list.append(x_train_men)
        x_test_men_list.append(x_test_men)

        y_train_women_list.append(y_train_women)
        y_test_women_list.append(y_test_women)
        y_train_men_list.append(y_train_men)
        y_test_men_list.append(y_test_men)
    
    else:
        df_train_women_list = df_train_women_list[0][columns_to_include_women]
        df_test_women_list = df_test_women_list[0][columns_to_include_women]
        df_train_men_list = df_train_men_list[0][columns_to_include_men]
        df_test_men_list = df_test_men_list[0][columns_to_include_men]

        x_train_women, y_train_women = x_y_from_data_frame(df_train_women_list)
        x_test_women, y_test_women = x_y_from_data_frame(df_test_women_list)
        x_train_men, y_train_men = x_y_from_data_frame(df_train_men_list)
        x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list)

        x_train_women, y_train_women = clear_na_from_x_y(x_train_women.copy(), y_train_women.copy())
        x_test_women, y_test_women = clear_na_from_x_y(x_test_women.copy(), y_test_women.copy())
        x_train_men, y_train_men = clear_na_from_x_y(x_train_men.copy(), y_train_men.copy())
        x_test_men, y_test_men = clear_na_from_x_y(x_test_men.copy(), y_test_men.copy())

In [35]:
j = year_range[6]

In [38]:
range(j, len(year_range))

range(2017, 12)

In [40]:
[year_range[a] for a in range(6+1, len(year_range))]

[2018, 2019, 2022, 2023, 2024]

In [47]:
[year_range[i] for i in range(6+1, len(year_range))]

[2018, 2019, 2022, 2023, 2024]

In [55]:
def objective_lr(trial, x_train, y_train, x_val, y_val):
    C = trial.suggest_loguniform("C", 1e-4, 10)
    
    # Define the model pipeline with scaling
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(C=C, solver="liblinear", max_iter=1000))
    ])
    
    # Train the model
    model.fit(x_train, y_train)
    
    # Predict probabilities
    y_pred = model.predict_proba(x_val)[:, 1]
    
    # Return Brier Score as the optimization metric
    return brier_score_loss(y_val, y_pred)


###################################################### WOMEN

# Split data for tuning
x_train_women_tr, x_val_women, y_train_women_tr, y_val_women = train_test_split(
    x_train_women, y_train_women, test_size=0.2, random_state=42
)

# Run Optuna optimization
study = optuna.create_study(direction="minimize")
study.optimize(lambda trial: objective_lr(trial, x_train_women_tr, y_train_women_tr, x_val_women, y_val_women), n_trials=50)

# Get best parameters
best_params_women_lr = study.best_params
print("Best Logistic Regression Params (Women):", best_params_women_lr)

# Train final Logistic Regression model with best params
best_lr_women = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(C=best_params_women_lr["C"], solver="liblinear", max_iter=1000))
])
best_lr_women.fit(x_train_women, y_train_women)

# Predict on test set
y_pred_women_lr = best_lr_women.predict_proba(x_test_women)[:, 1]

# Compute Brier Score
brier_women_lr = brier_score_loss(y_test_women, y_pred_women_lr)
print("Best Logistic Regression Brier Score (Women):", brier_women_lr)


###################################################### MEN

# Split data for tuning
x_train_men_tr, x_val_men, y_train_men_tr, y_val_men = train_test_split(
    x_train_men, y_train_men, test_size=0.2, random_state=42
)

# Run Optuna optimization
study = optuna.create_study(direction="minimize")
study.optimize(lambda trial: objective_lr(trial, x_train_men_tr, y_train_men_tr, x_val_men, y_val_men), n_trials=50)

# Get best parameters
best_params_men_lr = study.best_params
print("Best Logistic Regression Params (Men):", best_params_men_lr)

# Train final Logistic Regression model with best params
best_lr_men = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(C=best_params_men_lr["C"], solver="liblinear", max_iter=1000))
])
best_lr_men.fit(x_train_men, y_train_men)

# Predict on test set
y_pred_men_lr = best_lr_men.predict_proba(x_test_men)[:, 1]

# Compute Brier Score
brier_men_lr = brier_score_loss(y_test_men, y_pred_men_lr)
print("Best Logistic Regression Brier Score (Men):", brier_men_lr)


[I 2025-03-19 13:46:19,569] A new study created in memory with name: no-name-cc92e51c-e325-407e-8568-6597939eb155
C:\Users\Sebastian\AppData\Local\Temp\ipykernel_7980\932237102.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform("C", 1e-4, 10)
[I 2025-03-19 13:46:19,600] Trial 0 finished with value: 0.1368169319197489 and parameters: {'C': 0.022569896441928264}. Best is trial 0 with value: 0.1368169319197489.
C:\Users\Sebastian\AppData\Local\Temp\ipykernel_7980\932237102.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform("C", 1e-4, 10)
[I 2025-03-19 13:46:19,621] Trial 1 finished with value: 0.1400367430807521 a

[I 2025-03-19 13:46:19,882] Trial 16 finished with value: 0.140036086920646 and parameters: {'C': 2.337564660992037}. Best is trial 9 with value: 0.1367093337119125.
C:\Users\Sebastian\AppData\Local\Temp\ipykernel_7980\932237102.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform("C", 1e-4, 10)
[I 2025-03-19 13:46:19,916] Trial 17 finished with value: 0.14486113724667296 and parameters: {'C': 0.002448120335813451}. Best is trial 9 with value: 0.1367093337119125.
C:\Users\Sebastian\AppData\Local\Temp\ipykernel_7980\932237102.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform("C", 1e-4, 10)
[I 2025-03-19 13:46:19,

C:\Users\Sebastian\AppData\Local\Temp\ipykernel_7980\932237102.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform("C", 1e-4, 10)
[I 2025-03-19 13:46:20,225] Trial 34 finished with value: 0.1367327000070382 and parameters: {'C': 0.018803742712865553}. Best is trial 19 with value: 0.136703132331818.
C:\Users\Sebastian\AppData\Local\Temp\ipykernel_7980\932237102.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform("C", 1e-4, 10)
[I 2025-03-19 13:46:20,242] Trial 35 finished with value: 0.13860269644048157 and parameters: {'C': 0.005543343524306923}. Best is trial 19 with value: 0.136703132331818.
C:\Users\Sebastian\

C:\Users\Sebastian\AppData\Local\Temp\ipykernel_7980\932237102.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform("C", 1e-4, 10)
[I 2025-03-19 13:46:20,522] Trial 1 finished with value: 0.2271914961719112 and parameters: {'C': 0.00010370262635775421}. Best is trial 0 with value: 0.20148422351679887.
C:\Users\Sebastian\AppData\Local\Temp\ipykernel_7980\932237102.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform("C", 1e-4, 10)
[I 2025-03-19 13:46:20,536] Trial 2 finished with value: 0.17652519987065626 and parameters: {'C': 0.04615320155240535}. Best is trial 2 with value: 0.17652519987065626.
C:\Users\Sebastian

Best Logistic Regression Params (Women): {'C': 0.015822175961415122}
Best Logistic Regression Brier Score (Women): 0.16210798811477947


C:\Users\Sebastian\AppData\Local\Temp\ipykernel_7980\932237102.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform("C", 1e-4, 10)
[I 2025-03-19 13:46:20,682] Trial 8 finished with value: 0.1770126564546786 and parameters: {'C': 0.1911007114943728}. Best is trial 2 with value: 0.17652519987065626.
C:\Users\Sebastian\AppData\Local\Temp\ipykernel_7980\932237102.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform("C", 1e-4, 10)
[I 2025-03-19 13:46:20,702] Trial 9 finished with value: 0.17725074584787257 and parameters: {'C': 0.2932782141996007}. Best is trial 2 with value: 0.17652519987065626.
C:\Users\Sebastian\AppD

C:\Users\Sebastian\AppData\Local\Temp\ipykernel_7980\932237102.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform("C", 1e-4, 10)
[I 2025-03-19 13:46:21,149] Trial 25 finished with value: 0.17773477670503088 and parameters: {'C': 0.8314399471545122}. Best is trial 2 with value: 0.17652519987065626.
C:\Users\Sebastian\AppData\Local\Temp\ipykernel_7980\932237102.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform("C", 1e-4, 10)
[I 2025-03-19 13:46:21,169] Trial 26 finished with value: 0.17652456363969546 and parameters: {'C': 0.04427723146406927}. Best is trial 26 with value: 0.17652456363969546.
C:\Users\Sebastian

C:\Users\Sebastian\AppData\Local\Temp\ipykernel_7980\932237102.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform("C", 1e-4, 10)
[I 2025-03-19 13:46:21,588] Trial 42 finished with value: 0.17670700609851173 and parameters: {'C': 0.10164375433774743}. Best is trial 26 with value: 0.17652456363969546.
C:\Users\Sebastian\AppData\Local\Temp\ipykernel_7980\932237102.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform("C", 1e-4, 10)
[I 2025-03-19 13:46:21,621] Trial 43 finished with value: 0.17652598554556823 and parameters: {'C': 0.047231993534291014}. Best is trial 26 with value: 0.17652456363969546.
C:\Users\Sebast

Best Logistic Regression Params (Men): {'C': 0.04427723146406927}
Best Logistic Regression Brier Score (Men): 0.2077845929872274


In [58]:
columns_to_include_women = [
 'Season',
 'DayNum',
 'T1_TeamID',
 'T1_Score',
 'T2_TeamID',
 'T2_Score',
 'location',
                      
#  'T1_Score_mean',
 'T1_FGM',
 'T1_FGA',
 'T1_FGM3',
 'T1_FGA3',
#  'T1_FTM',
#  'T1_FTA',
 'T1_OR',
#  'T1_DR',
 'T1_Ast',
 'T1_TO',
 'T1_Stl',
#  'T1_Blk',
 'T1_PF',
                      
#  'T1_opponent_Score',
 'T1_opponent_FGM',
 'T1_opponent_FGA',
 'T1_opponent_FGM3',
 'T1_opponent_FGA3',
#  'T1_opponent_FTM',
#  'T1_opponent_FTA',
 'T1_opponent_OR',
#  'T1_opponent_DR',
 'T1_opponent_Ast',
 'T1_opponent_TO',
 'T1_opponent_Stl',
#  'T1_opponent_Blk',
 'T1_opponent_PF',
 'T1_PointDiff',
                      
#  'T2_Score_mean',
 'T2_FGM',
 'T2_FGA',
 'T2_FGM3',
 'T2_FGA3',
#  'T2_FTM',
#  'T2_FTA',
 'T2_OR',
#  'T2_DR',
 'T2_Ast',
 'T2_TO',
 'T2_Stl',
#  'T2_Blk',
 'T2_PF',
                      
#  'T2_opponent_Score',
 'T2_opponent_FGM',
 'T2_opponent_FGA',
 'T2_opponent_FGM3',
 'T2_opponent_FGA3',
#  'T2_opponent_FTM',
#  'T2_opponent_FTA',
 'T2_opponent_OR',
#  'T2_opponent_DR',
 'T2_opponent_Ast',
 'T2_opponent_TO',
 'T2_opponent_Stl',
#  'T2_opponent_Blk',
 'T2_opponent_PF',
 'T2_PointDiff',
                      
 'T1_win_ratio_14d',
 'T2_win_ratio_14d',
 'T1_seed',
 'T2_seed',
 'Seed_diff',
 'TeamELO',
#  'CoachELO'
]
columns_to_include_men = [
 'Season',
 'DayNum',
 'T1_TeamID',
 'T1_Score',
 'T2_TeamID',
 'T2_Score',
 'location',
                      
#  'T1_Score_mean',
 'T1_FGM',
 'T1_FGA',
 'T1_FGM3',
 'T1_FGA3',
#  'T1_FTM',
#  'T1_FTA',
 'T1_OR',
#  'T1_DR',
 'T1_Ast',
 'T1_TO',
 'T1_Stl',
#  'T1_Blk',
 'T1_PF',
                      
#  'T1_opponent_Score',
 'T1_opponent_FGM',
 'T1_opponent_FGA',
 'T1_opponent_FGM3',
 'T1_opponent_FGA3',
#  'T1_opponent_FTM',
#  'T1_opponent_FTA',
 'T1_opponent_OR',
#  'T1_opponent_DR',
 'T1_opponent_Ast',
 'T1_opponent_TO',
 'T1_opponent_Stl',
#  'T1_opponent_Blk',
 'T1_opponent_PF',
 'T1_PointDiff',
                      
#  'T2_Score_mean',
 'T2_FGM',
 'T2_FGA',
 'T2_FGM3',
 'T2_FGA3',
#  'T2_FTM',
#  'T2_FTA',
 'T2_OR',
#  'T2_DR',
 'T2_Ast',
 'T2_TO',
 'T2_Stl',
#  'T2_Blk',
 'T2_PF',
                      
#  'T2_opponent_Score',
 'T2_opponent_FGM',
 'T2_opponent_FGA',
 'T2_opponent_FGM3',
 'T2_opponent_FGA3',
#  'T2_opponent_FTM',
#  'T2_opponent_FTA',
 'T2_opponent_OR',
#  'T2_opponent_DR',
 'T2_opponent_Ast',
 'T2_opponent_TO',
 'T2_opponent_Stl',
#  'T2_opponent_Blk',
 'T2_opponent_PF',
 'T2_PointDiff',
                      
 'T1_win_ratio_14d',
 'T2_win_ratio_14d',
 'T1_seed',
 'T2_seed',
 'Seed_diff',
 'TeamELO',
 'CoachELO'
]
print('All columns with ELO')
year_range = [i for i in range(2011, 2020)] + [i for i in range(2022, 2025)]
# ----------------------------------------------------------
# PARAMETERS TO CHANGE
start_season = 2003 # from which ponit should we begin creating data
season_years_list  = [[[i] for i in range(start_season, year+1)] for year in year_range] # at which seasons to look at when calculating team's stats
days_back = 150 # how many days back from the start of tourney to calculate team's stats per season
location_multiplier = [0.9, 1.1] # home penalty, away bonus ex.
win_ratio_days_back = 14 # how many days back from the tourney do we calculate win ratio
df_train_women_list, df_test_women_list, df_train_men_list, df_test_men_list = brier_for_all_years(year_range, season_years_list)
# ----------------------------------------------------------
x_train_women_list = []
x_test_women_list = []
x_train_men_list = []
x_test_men_list = []
y_train_women_list = []
y_test_women_list = []
y_train_men_list = []
y_test_men_list = []
brier_women_list = []
brier_men_list = []
brier_all_list = []
for i in range(len(year_range)):
    df_train_women_list[i] = df_train_women_list[i][columns_to_include_women]
    df_test_women_list[i] = df_test_women_list[i][columns_to_include_women]
    df_train_men_list[i] = df_train_men_list[i][columns_to_include_men]
    df_test_men_list[i] = df_test_men_list[i][columns_to_include_men]
    
    x_train_women, y_train_women = x_y_from_data_frame(df_train_women_list[i])
    x_test_women, y_test_women = x_y_from_data_frame(df_test_women_list[i])
    x_train_men, y_train_men = x_y_from_data_frame(df_train_men_list[i])
    x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list[i])

    x_train_women, y_train_women = clear_na_from_x_y(x_train_women.copy(), y_train_women.copy())
    x_test_women, y_test_women = clear_na_from_x_y(x_test_women.copy(), y_test_women.copy())
    x_train_men, y_train_men = clear_na_from_x_y(x_train_men.copy(), y_train_men.copy())
    x_test_men, y_test_men = clear_na_from_x_y(x_test_men.copy(), y_test_men.copy())
    
    x_train_women_list.append(x_train_women)
    x_test_women_list.append(x_test_women)
    x_train_men_list.append(x_train_men)
    x_test_men_list.append(x_test_men)
    
    y_train_women_list.append(y_train_women)
    y_test_women_list.append(y_test_women)
    y_train_men_list.append(y_train_men)
    y_test_men_list.append(y_test_men)
    
# -------------------------------------------------------

for i in range(len(year_range)):
    print("Predictions for season ", year_range[i])
    x_train_women = x_train_women_list[i]
    y_train_women = y_train_women_list[i]
    x_test_women = x_test_women_list[i]
    y_test_women = y_test_women_list[i]
    
    best_lr_women = LogisticRegression(**best_params_women_lr)
    best_lr_women.fit(x_train_women, y_train_women)
    
    y_prob_women = best_lr_women.predict_proba(x_test_women)[:, 1]
    score = brier_score_loss(y_test_women, y_prob_women)
    brier_women_list.append(score)
    print('LogisticRegressionCV for women trained on women', score)
    # ----------------------------------------------
    x_train_men = x_train_men_list[i]
    y_train_men = y_train_men_list[i]
    x_test_men = x_test_men_list[i]
    y_test_men = y_test_men_list[i]
    
    best_lr_men = LogisticRegression(**best_params_men_lr)
    best_lr_men.fit(x_train_men, y_train_men)
    
    y_prob_men = best_lr_men.predict_proba(x_test_men)[:, 1]
    score = brier_score_loss(y_test_men, y_prob_men)
    brier_men_list.append(score)
    print('LogisticRegressionCV for men trained on men', score)
    # ----------------------------------------------
    y_prob = np.concatenate((y_prob_women, y_prob_men))
    y_test = np.concatenate((y_test_women, y_test_men))
    score = brier_score_loss(y_test, y_prob)
    brier_all_list.append(score)
    print('LogisticRegressionCV for all trained on women and men separately', score, '<---')
    print()
    # ----------------------------------------------
print()
print('The mean score when trained on women and tested on women was: ', np.mean(brier_women_list), 
      'with a std of ', np.std(brier_women_list))
print('The mean score when trained on men and tested on men was: ', np.mean(brier_men_list), 
      'with a std of ', np.std(brier_men_list))
print('The mean score when trained separately and tested on both was: ', np.mean(brier_all_list), 
      'with a std of ', np.std(brier_all_list))

All columns with ELO
Predictions for season  2011
LogisticRegressionCV for women trained on women 0.13708332067272705
LogisticRegressionCV for men trained on men 0.2330086587288496
LogisticRegressionCV for all trained on women and men separately 0.18652176413242097 <---

Predictions for season  2012
LogisticRegressionCV for women trained on women 0.11733945405251447
LogisticRegressionCV for men trained on men 0.19704398632394646
LogisticRegressionCV for all trained on women and men separately 0.15900318683076303 <---

Predictions for season  2013


C:\Users\Sebastian\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:814: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\Sebastian\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:814: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
 

LogisticRegressionCV for women trained on women 0.16967314739131975
LogisticRegressionCV for men trained on men 0.20880887482215393
LogisticRegressionCV for all trained on women and men separately 0.18984309922105733 <---

Predictions for season  2014
LogisticRegressionCV for women trained on women 0.13467617165571713
LogisticRegressionCV for men trained on men 0.21694517727381207
LogisticRegressionCV for all trained on women and men separately 0.17707635147427375 <---

Predictions for season  2015
LogisticRegressionCV for women trained on women 0.13772002126340568
LogisticRegressionCV for men trained on men 0.16316738099707692
LogisticRegressionCV for all trained on women and men separately 0.15083519897229783 <---

Predictions for season  2016
LogisticRegressionCV for women trained on women 0.2013681006978301


C:\Users\Sebastian\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:814: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\Sebastian\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:814: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
 

LogisticRegressionCV for men trained on men 0.19822185997040204
LogisticRegressionCV for all trained on women and men separately 0.19971218452549955 <---

Predictions for season  2017
LogisticRegressionCV for women trained on women 0.15446668372293454
LogisticRegressionCV for men trained on men 0.18394681735050616
LogisticRegressionCV for all trained on women and men separately 0.1696602910540676 <---

Predictions for season  2018
LogisticRegressionCV for women trained on women 0.1666417865272158


...

C:\Users\Sebastian\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:814: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\Sebastian\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:814: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
 

LogisticRegressionCV for women trained on women 0.1648182330619324
LogisticRegressionCV for men trained on men 0.24140724650936918
LogisticRegressionCV for all trained on women and men separately 0.20339640279841906 <---

Predictions for season  2023
LogisticRegressionCV for women trained on women 0.16256018438777503


C:\Users\Sebastian\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:814: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\Sebastian\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:814: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
 

LogisticRegressionCV for men trained on men 0.2045611105016789
LogisticRegressionCV for all trained on women and men separately 0.18416935651884153 <---

Predictions for season  2024
LogisticRegressionCV for women trained on women 0.12896757292420966
LogisticRegressionCV for men trained on men 0.19694737676080556
LogisticRegressionCV for all trained on women and men separately 0.16295747484250758 <---


The mean score when trained on women and tested on women was:  0.150650972358275 with a std of  0.022477313825635652
The mean score when trained on men and tested on men was:  0.20162398165058748 with a std of  0.021760260490081323
The mean score when trained separately and tested on both was:  0.17680275375624885 with a std of  0.017040003450807254


C:\Users\Sebastian\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:814: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
